# CS781 Phase 5 — Legal Statute Identification

**Kaggle T4 ×2 · seven adaptation submissions · Macro-F1 · your original Phase-1 models**

This is multi-label classification: each case receives a set of integer statute IDs. The notebook compares full fine-tuning, partial tuning, compulsory LoRMA+, adapters, soft prompts, gradual unfreezing and a smaller knowledge-distilled student. All four original foundation families are used. General instructions require at least six distinct submissions; this notebook produces seven candidates and reports their prediction hashes.

The speed strategy shares complete-document frozen feature banks and compatible cached heads, then adapts each encoder on a small set of evidence windows. Dev selects asymmetric loss versus weighted BCE, checkpoints, fusion and shrunk label thresholds. No external pretrained weights or LLM services are used. Supplied test labels, if present, are discarded from model data.

The official Phase-5 deadline is **October 26, 2026, 11:59 PM IST**. Scores and total runtime depend on your data and Phase-1 checkpoints; no target score is guaranteed. This notebook is AI-assisted: preserve attribution and follow the disclosure/course-use rules in the [capstone brief](https://docs.google.com/document/d/1olqoT5KPhf5CSJLBNjXGyKyyyQDBr6jcnWqPUh5cNk4/edit).

In [ ]:
import os,sys,subprocess,importlib.util
os.environ['TOKENIZERS_PARALLELISM']='false'
os.environ['OMP_NUM_THREADS']='1'
os.environ.setdefault('HF_HUB_CACHE','/tmp/capstone_phase5_hf')
requirements={'tokenizers':'tokenizers>=0.20,<1','huggingface_hub':'huggingface_hub>=0.27,<3',
              'sklearn':'scikit-learn>=1.4,<2','matplotlib':'matplotlib>=3.8,<4',
              'joblib':'joblib>=1.3,<2','scipy':'scipy>=1.11,<2'}
missing=[spec for name,spec in requirements.items() if importlib.util.find_spec(name) is None]
if missing:subprocess.run([sys.executable,'-m','pip','install','-q',*missing],check=True)
import torch,numpy as np
if tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2])<(2,3):
    raise RuntimeError('Use a Kaggle PyTorch 2.3+ image; retain its installed CUDA/PyTorch pairing.')
torch.set_num_threads(1)
print('PyTorch:',torch.__version__)

## Configure inputs and training

Select **GPU T4 ×2** in Kaggle. Attach your **phase-1-outputs** dataset. The Phase-5 brief requires original Phase-1 initialization, so Phase-4 task weights are not substituted. The default path below is the exact Phase-1 root supplied earlier.

`DATA_ROOT=None` downloads the official Phase-5 files using an `HF_TOKEN` Kaggle secret if gated. Alternatively attach the official files and set `DATA_ROOT`. Keep project, data, paths and training settings fixed to resume. Two adaptation epochs and three evidence windows are the compute-conscious defaults; raise them in a new project if measured dev results justify more compute. No automatic time cutoff changes the experiment.

In [ ]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import json,csv,zipfile,shutil,importlib
from IPython.display import display,FileLink
WORKING=Path('/kaggle/working')
PROJECT=WORKING/'capstone_phase5'
CODE=PROJECT/'code';CODE.mkdir(parents=True,exist_ok=True)
PHASE1_ROOT=Path('/kaggle/input/datasets/hardik4tiwari/phase-1-outputs/capstone_phase1')
PARENT_OVERRIDES={}
DATA_ROOT=None  # Or Path('/kaggle/input/YOUR-DATA/Phase_5_LSI').
DATA_REVISION=None
SETTINGS={
    'gpus':2,'chunk_tokens':384,'overlap':48,'chunk_batch':16,
    'head_dim':192,'head_batch':32,'head_epochs':12,'head_patience':3,
    'head_losses':['asl','bce'],
    'batch_docs':4,'accumulation':2,'adapt_epochs':2,'adapt_patience':2,
    'adapt_chunks':3,'student_layers':4,
}
RUN_BENCHMARK=True
RUN_GPU_SMOKE=True
USE_LEXICAL_FUSION=True
LEXICAL_MAX_FEATURES=80000
MAKE_BONUS_ENSEMBLE=True
EVALUATE_SUPPLIED_TEST_LABELS=True  # Reports supplied labels after predictions are frozen; skips absent truth.
MAKE_MODEL_ARCHIVE=True
MAKE_FULL_WORKING_ARCHIVE=False  # Optional; can duplicate many GB of recomputable caches.
if torch.cuda.device_count()!=2:raise RuntimeError('Select Kaggle GPU T4 x2 before Run All.')
for i in range(2):
    gpu=torch.cuda.get_device_properties(i)
    print(i,gpu.name,round(gpu.total_memory/1024**3,2),'GiB; capability',gpu.major,gpu.minor)
sys.path.insert(0,str(CODE))
print('Current IST:',datetime.now(ZoneInfo('Asia/Kolkata')).isoformat(timespec='minutes'))
print('Original competition deadline: 2026-10-26 23:59 IST')

## Exact input structure

```text
capstone_phase1/
    tokenizer.json
    runs/main_decoder/best_model.pt
    runs/main_encoder_decoder/best_model.pt
    runs/main_local_gqa/best_model.pt
    runs/main_hybrid/best_model.pt

Phase_5_LSI/
    Train/train.json
    Train/dev.json
    Test/test.json
```

Train/dev records normally contain `{"id":"case_id","text":"...","labels":[1,7,23]}`. Lists, mappings and column exports are supported. Raw case IDs are retained. Label IDs are not renumbered. Repeated cases are audited and aggregated; malformed IDs or labels fail before training. Supplied test labels are ignored by the training/inference parser.

## Write the complete runtime

All runtime source and the unchanged official Phase-5 evaluator are embedded below. The foundation definitions retain the trained Phase-1 rotary layout. The hybrid LSTM remains outside checkpoint recomputation. There are no external pretrained-model downloads.

In [ ]:
MODULE_SOURCE=r'''
"""Four randomly initialized legal language models; no downloaded model weights."""
import math
from dataclasses import dataclass, asdict
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint


@dataclass
class ModelConfig:
    architecture: str = "decoder"
    vocab_size: int = 32768
    dim: int = 512
    heads: int = 8
    kv_heads: int = 8
    layers: int = 8
    ff_dim: int = 2048
    encoder_layers: int = 0
    rnn_layers: int = 0
    local_window: int = 0
    max_positions: int = 4096
    dropout: float = 0.0
    gradient_checkpointing: bool = False


def architecture_config(name, **overrides):
    c = ModelConfig(architecture=name)
    if name == "decoder":
        pass
    elif name == "local_gqa":
        c.kv_heads, c.layers, c.local_window = 2, 9, 256
    elif name == "hybrid":
        c.layers, c.rnn_layers = 7, 2
    elif name == "encoder_decoder":
        c.layers, c.encoder_layers = 4, 3
    else:
        raise ValueError(f"Unknown architecture: {name}")
    for k, v in overrides.items():
        if not hasattr(c, k):
            raise ValueError(k)
        setattr(c, k, v)
    assert c.dim % c.heads == 0 and c.heads % c.kv_heads == 0
    assert (c.dim // c.heads) % 2 == 0
    return c


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        # Reduction in FP32 is useful with T4 FP16 training.
        y = x.float() * torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return y.to(x.dtype) * self.weight.to(x.dtype)


class Rotary(nn.Module):
    def __init__(self, head_dim, max_positions):
        super().__init__()
        inv = 1.0 / (10000 ** (torch.arange(0, head_dim, 2).float() / head_dim))
        angles = torch.outer(torch.arange(max_positions).float(), inv)
        self.register_buffer("cos", angles.cos(), persistent=False)
        self.register_buffer("sin", angles.sin(), persistent=False)

    def forward(self, x, offset=0):
        n = x.size(-2)
        if offset + n > self.cos.size(0):
            raise ValueError("Position limit exceeded; increase max_positions in the model config.")
        cos = self.cos[offset:offset + n].to(x.dtype)[None, None]
        sin = self.sin[offset:offset + n].to(x.dtype)[None, None]
        a, b = x[..., 0::2], x[..., 1::2]
        return torch.stack((a * cos - b * sin, a * sin + b * cos), dim=-1).flatten(-2)


class Attention(nn.Module):
    def __init__(self, c, causal=True, cross=False):
        super().__init__()
        self.heads, self.kv_heads = c.heads, c.kv_heads
        self.hd = c.dim // c.heads
        self.q = nn.Linear(c.dim, c.dim, bias=False)
        self.k = nn.Linear(c.dim, self.kv_heads * self.hd, bias=False)
        self.v = nn.Linear(c.dim, self.kv_heads * self.hd, bias=False)
        self.out = nn.Linear(c.dim, c.dim, bias=False)
        self.rotary = None if cross else Rotary(self.hd, c.max_positions)
        self.causal, self.cross = causal, cross
        self.window = c.local_window if causal and not cross else 0
        self.dropout = c.dropout

    def forward(self, x, memory=None, past=None, position=0, use_cache=False):
        b, t, _ = x.shape
        q = self.q(x).view(b, t, self.heads, self.hd).transpose(1, 2)
        if self.cross and past is not None:
            k, v = past  # Encoder K/V projections are also cached during decoding.
        else:
            source = x if memory is None else memory
            s = source.size(1)
            k = self.k(source).view(b, s, self.kv_heads, self.hd).transpose(1, 2)
            v = self.v(source).view(b, s, self.kv_heads, self.hd).transpose(1, 2)
            if self.rotary is not None:
                k = self.rotary(k, position)
            if past is not None:
                k, v = torch.cat((past[0], k), -2), torch.cat((past[1], v), -2)
        if self.rotary is not None:
            q = self.rotary(q, position)
        new_cache = (k, v) if use_cache else None
        # Portable GQA: project/cache fewer KV heads, repeat only for the SDPA call.
        # This is not a block-sparse implementation or a guaranteed training speedup.
        if self.kv_heads != self.heads:
            repeats = self.heads // self.kv_heads
            k = k.repeat_interleave(repeats, dim=1)
            v = v.repeat_interleave(repeats, dim=1)
        mask, is_causal = None, False
        if self.causal:
            s = k.size(-2)
            if self.window or (past is not None and t > 1):
                qi = torch.arange(s - t, s, device=x.device)[:, None]
                ki = torch.arange(s, device=x.device)[None, :]
                mask = ki <= qi
                if self.window:
                    mask = mask & ((qi - ki) < self.window)
            elif past is None:
                is_causal = True
            # Cached single-token query must see all prior keys. is_causal=True
            # here would incorrectly use the upper-left non-square mask.
        y = F.scaled_dot_product_attention(
            q, k, v, attn_mask=mask, is_causal=is_causal,
            dropout_p=self.dropout if self.training else 0.0,
        )
        y = self.out(y.transpose(1, 2).contiguous().view(b, t, -1))
        return y, new_cache


class SwiGLU(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.gate = nn.Linear(c.dim, c.ff_dim, bias=False)
        self.up = nn.Linear(c.dim, c.ff_dim, bias=False)
        self.down = nn.Linear(c.ff_dim, c.dim, bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))


class Block(nn.Module):
    def __init__(self, c, encoder=False):
        super().__init__()
        self.attn_norm, self.ff_norm = RMSNorm(c.dim), RMSNorm(c.dim)
        self.attn = Attention(c, causal=not encoder)
        self.ff = SwiGLU(c)
        self.drop = nn.Dropout(c.dropout)

    def forward(self, x, past=None, position=0, use_cache=False):
        y, cache = self.attn(self.attn_norm(x), past=past, position=position, use_cache=use_cache)
        x = x + self.drop(y)
        x = x + self.drop(self.ff(self.ff_norm(x)))
        return (x, cache) if use_cache else x


class CrossBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.self_norm, self.cross_norm, self.ff_norm = (RMSNorm(c.dim) for _ in range(3))
        self.self_attn = Attention(c)
        self.cross_attn = Attention(c, causal=False, cross=True)
        self.ff, self.drop = SwiGLU(c), nn.Dropout(c.dropout)

    def forward(self, x, memory, past=None, cross_past=None, position=0, use_cache=False):
        y, cache = self.self_attn(self.self_norm(x), past=past, position=position, use_cache=use_cache)
        x = x + self.drop(y)
        y, cross_cache = self.cross_attn(self.cross_norm(x), memory=memory, past=cross_past, use_cache=use_cache)
        x = x + self.drop(y)
        x = x + self.drop(self.ff(self.ff_norm(x)))
        return (x, cache, cross_cache) if use_cache else x


class LegalLM(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.config = c
        self.embedding = nn.Embedding(c.vocab_size, c.dim)
        # Output projection uses embedding.weight; there is no second vocabulary matrix.
        if c.architecture == "encoder_decoder":
            self.encoder = nn.ModuleList(Block(c, encoder=True) for _ in range(c.encoder_layers))
            self.encoder_norm = RMSNorm(c.dim)
            self.blocks = nn.ModuleList(CrossBlock(c) for _ in range(c.layers))
        else:
            self.blocks = nn.ModuleList(Block(c) for _ in range(c.layers))
        if c.rnn_layers:
            self.rnn = nn.LSTM(c.dim, c.dim, num_layers=c.rnn_layers, batch_first=True)
            self.rnn_norm = RMSNorm(c.dim)
        self.final_norm = RMSNorm(c.dim)
        self.apply(self._init)
        # Scale residual projections to keep initial residual variance controlled.
        std = 0.02 / math.sqrt(2 * c.layers)
        for name, p in self.named_parameters():
            if name.endswith(("attn.out.weight", "self_attn.out.weight", "cross_attn.out.weight", "ff.down.weight")):
                nn.init.normal_(p, mean=0.0, std=std)

    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        elif isinstance(m, nn.LSTM):
            for name, p in m.named_parameters():
                if "weight" in name:
                    nn.init.normal_(p, mean=0.0, std=0.02)
                else:
                    nn.init.zeros_(p)

    def encode(self, source_ids):
        x = self.embedding(source_ids)
        for block in self.encoder:
            if self.config.gradient_checkpointing and self.training:
                x = checkpoint(block, x, use_reentrant=False)
            else:
                x = block(x)
        return self.encoder_norm(x)

    def forward(self, input_ids, source_ids=None, state=None, use_cache=False):
        state = state or {}
        position = state.get("position", 0)
        x = self.embedding(input_ids)
        new_state = {"position": position + input_ids.size(1), "self": []}
        if self.config.rnn_layers:
            y, recurrent_state = self.rnn(x, state.get("rnn"))
            x = x + self.rnn_norm(y)
            new_state["rnn"] = recurrent_state
        if self.config.architecture == "encoder_decoder":
            memory = state.get("memory")
            if memory is None:
                if source_ids is None:
                    raise ValueError("encoder_decoder requires source_ids on the first call.")
                memory = self.encode(source_ids)
            new_state.update(memory=memory, cross=[])
            for i, block in enumerate(self.blocks):
                if use_cache:
                    p = state.get("self", [None] * len(self.blocks))[i]
                    cp = state.get("cross", [None] * len(self.blocks))[i]
                    x, cache, cc = block(x, memory, p, cp, position, True)
                    new_state["self"].append(cache)
                    new_state["cross"].append(cc)
                elif self.config.gradient_checkpointing and self.training:
                    x = checkpoint(block, x, memory, use_reentrant=False)
                else:
                    x = block(x, memory)
        else:
            for i, block in enumerate(self.blocks):
                if use_cache:
                    p = state.get("self", [None] * len(self.blocks))[i]
                    x, cache = block(x, p, position, True)
                    new_state["self"].append(cache)
                elif self.config.gradient_checkpointing and self.training:
                    x = checkpoint(block, x, use_reentrant=False)
                else:
                    x = block(x)
        x = self.final_norm(x)
        return (x, new_state) if use_cache else x

    def project(self, hidden):
        return F.linear(hidden, self.embedding.weight)


def parameter_count(model):
    # PyTorch de-duplicates shared Parameter objects here.
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def parameter_formula(c):
    d, h, kv = c.dim, c.dim // c.heads, c.kv_heads
    attention = 2 * d * d + 2 * d * kv * h
    ff = 3 * d * c.ff_dim
    total = c.vocab_size * d + d
    if c.architecture == "encoder_decoder":
        total += c.encoder_layers * (attention + ff + 2 * d) + d
        total += c.layers * (2 * attention + ff + 3 * d)
    else:
        total += c.layers * (attention + ff + 2 * d)
    if c.rnn_layers:
        total += c.rnn_layers * (8 * d * d + 8 * d) + d
    return total


def make_batch(blocks, architecture, prefix_fraction=0.80, objective="full"):
    # blocks has T+1 tokens. The target suffix is identical across architectures.
    t = blocks.size(1) - 1
    split = max(1, min(t, int(t * prefix_fraction)))
    if architecture == "encoder_decoder":
        source = blocks[:, :split]
        inputs, targets = blocks[:, split - 1:-1], blocks[:, split:]
    else:
        source = None
        inputs, targets = blocks[:, :-1], blocks[:, 1:].clone()
        if objective == "continuation":
            targets[:, :split - 1] = -100
    return inputs, targets, source


def language_model_loss(hidden, targets, weight, chunk_tokens=256, checkpoint_chunks=False):
    """Sum/token count lets validation aggregate the true token-weighted NLL."""
    hidden, targets = hidden.reshape(-1, hidden.size(-1)), targets.reshape(-1)
    total = hidden.new_zeros((), dtype=torch.float32)
    count = (targets != -100).sum()
    def loss_chunk(h, y, w):
        # Keep logits in autocast precision until the bounded FP32 CE conversion.
        return F.cross_entropy(F.linear(h, w).float(), y, ignore_index=-100, reduction="sum")
    for i in range(0, targets.numel(), chunk_tokens):
        h, y = hidden[i:i + chunk_tokens], targets[i:i + chunk_tokens]
        if checkpoint_chunks and torch.is_grad_enabled():
            total = total + checkpoint(loss_chunk, h, y, weight, use_reentrant=False)
        else:
            total = total + loss_chunk(h, y, weight)
    return total / count.clamp_min(1), total.detach(), count.detach()


if __name__ == "__main__":
    for name in ("decoder", "encoder_decoder", "local_gqa", "hybrid"):
        c = architecture_config(name)
        with torch.device("meta"):
            model = LegalLM(c)
        n = parameter_count(model)
        assert n == parameter_formula(c) and 50_000_000 <= n <= 300_000_000
        print(name, f"{n:,}", asdict(c))
'''
(CODE/'legal_models.py').write_text(MODULE_SOURCE.lstrip('\n'),encoding='utf-8')
print('Wrote','legal_models.py')

In [ ]:
MODULE_SOURCE=r'''
"""Phase-5 schemas, leakage protection, all-token cache, sparse-label metrics and calibration."""
import hashlib,json,math,os,shutil,zipfile

from pathlib import Path

import numpy as np

import torch

from torch.utils.data import Dataset

from tokenizers import Tokenizer

from sklearn.metrics import f1_score,precision_score,recall_score,confusion_matrix

def digest(value):return hashlib.sha256(json.dumps(value,sort_keys=True,ensure_ascii=False,separators=(',',':')).encode()).hexdigest()

def file_hash(path):
    h=hashlib.sha256()
    with open(path,'rb') as handle:
        for chunk in iter(lambda:handle.read(1<<20),b''):h.update(chunk)
    return h.hexdigest()

def read_json(path):
    def unique(pairs):
        result={}
        for key,value in pairs:
            if key in result:raise ValueError(f'Duplicate JSON key: {key}')
            result[key]=value
        return result
    return json.loads(Path(path).read_text(encoding='utf-8-sig'),object_pairs_hook=unique)

def atomic_json(path,value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+'.tmp');tmp.write_text(json.dumps(value,ensure_ascii=False,indent=2),encoding='utf-8');tmp.replace(path)

def atomic_torch(path,value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+'.tmp');torch.save(value,tmp);tmp.replace(path)

def alias(row,names):
    keys=[name for name in names if name in row]
    if len(keys)>1 and any(row[k]!=row[keys[0]] for k in keys[1:]):raise ValueError(f'Conflicting fields: {keys}')
    return row[keys[0]] if keys else None

def shape(value):
    if isinstance(value,dict):return {str(k):type(v).__name__ for k,v in list(value.items())[:8]}
    return {'type':type(value).__name__,'length':len(value) if isinstance(value,list) else None}

def to_device(batch,device):return {k:v.to(device,non_blocking=True) if isinstance(v,torch.Tensor) else v for k,v in batch.items()}

import re,bisect
DATA_FILES=['Train/train.json','Train/dev.json','Test/test.json']

def records(value,split,audit=None):
    if isinstance(value,dict) and set(value) in ({'data'},{'records'},{'documents'}):value=next(iter(value.values()))
    if isinstance(value,list):rows=[(None,row) for row in value]
    elif isinstance(value,dict):
        texts=alias(value,('text','sentences','document'));ids=alias(value,('id','ids','case_id','doc_id'))
        labels=None if split=='test' else alias(value,('labels','label','gold','statutes','statute_ids'))
        if isinstance(ids,str) and texts is not None:rows=[(None,{'id':ids,'text':texts,'labels':labels})]
        elif isinstance(ids,list) and isinstance(texts,list):
            if len(ids)!=len(texts) or (labels is not None and (not isinstance(labels,list) or len(labels)!=len(ids))):raise ValueError(f'{split}: column lengths differ.')
            rows=[(None,{'id':did,'text':texts[i],'labels':None if labels is None else labels[i]}) for i,did in enumerate(ids)]
        elif isinstance(texts,dict):
            for column in (ids,labels):
                if column is not None and (not isinstance(column,dict) or set(column)!=set(texts)):raise ValueError('Column keys differ.')
            rows=[(None,{'id':key if ids is None else ids[key],'text':body,'labels':None if labels is None else labels[key]}) for key,body in texts.items()]
        else:rows=list(value.items())
    else:raise ValueError(f'{split}: unsupported input structure: {shape(value)}')
    grouped={};exact=set();copies=0;different=0
    for key,row in rows:
        if isinstance(row,str) and key is not None:row={'id':key,'text':row}
        if not isinstance(row,dict):raise ValueError(f'{split}: document must be an object.')
        did=alias(row,('id','case_id','doc_id'));did=key if did is None else did
        if not isinstance(did,str) or not did.strip():raise ValueError(f'{split}: original nonempty string IDs are required.')
        if key is not None and key!=did:raise ValueError('Mapping ID and explicit case ID differ.')
        text=alias(row,('text','sentences','document'))
        if isinstance(text,list) and all(isinstance(x,str) for x in text):text='\n'.join(text)
        if not isinstance(text,str):raise ValueError(f'{did}: expected text or an ordered sentence-string list.')
        gold=None if split=='test' else alias(row,('labels','label','gold','statutes','statute_ids'))
        if split!='test':
            if not isinstance(gold,list) or any(type(x) is not int or x<0 for x in gold):raise ValueError(f'{did}: labels must be a list of nonnegative integer statute IDs.')
            gold=sorted(set(gold))
        signature=(did,digest(text),None if gold is None else tuple(gold))
        if signature in exact:copies+=1;continue
        exact.add(signature)
        entry=grouped.setdefault(did,{'texts':[],'text_hashes':set(),'gold':set(),'rows':0,'label_sets':set()})
        entry['rows']+=1;sha=digest(text)
        if sha not in entry['text_hashes']:entry['text_hashes'].add(sha);entry['texts'].append(text)
        if gold is not None:entry['gold'].update(gold);entry['label_sets'].add(tuple(gold))
    result=[{'id':did,'source_id':did,'text':'\n\n'.join(v['texts']),
             'gold':None if split=='test' else sorted(v['gold']),'split':split} for did,v in grouped.items()]
    if not result:raise ValueError(f'{split}: empty split.')
    if audit is not None:audit.update(raw_rows=len(rows),case_count=len(result),exact_copies_removed=copies,
        repeated_cases_combined=sum(v['rows']>1 for v in grouped.values()),
        cases_with_different_label_sets=sum(len(v['label_sets'])>1 for v in grouped.values()),
        policy='One original ID per case; distinct texts concatenate in source order; supplied train/dev label sets union. Test labels are discarded.')
    return result

def load_records(root,return_audit=False):
    root=Path(root);audit={};all_rows={}
    for split,name in zip(('train','dev','test'),DATA_FILES):
        audit[split]={};all_rows[split]=records(read_json(root/name),split,audit[split])
    test_ids={r['id'] for r in all_rows['test']};dev_ids={r['id'] for r in all_rows['dev']}
    audit['dev_test_source_overlap_removed']=sorted(dev_ids&test_ids)
    all_rows['dev']=[r for r in all_rows['dev'] if r['id'] not in test_ids]
    held_ids=dev_ids|test_ids
    held_text={digest(' '.join(r['text'].split())) for s in ('dev','test') for r in all_rows[s]}
    removed=[r['id'] for r in all_rows['train'] if r['id'] in held_ids or digest(' '.join(r['text'].split())) in held_text]
    remove=set(removed);all_rows['train']=[r for r in all_rows['train'] if r['id'] not in remove]
    audit['training_held_out_cases_or_text_copies_removed']=removed
    if any(not all_rows[s] for s in ('train','dev','test')):raise ValueError('Leakage filtering left an empty split; inspect IDs and schema before training.')
    result=sum([all_rows[s] for s in ('train','dev','test')],[])
    if len({r['id'] for r in result})!=len(result):raise ValueError('IDs overlap after case filtering.')
    return (result,audit) if return_audit else result

def prepare_data(root,tokenizer_path,output,chunk_tokens=384,overlap=48):
    root=Path(root);output=Path(output);output.mkdir(parents=True,exist_ok=True)
    if not 0<=overlap<chunk_tokens:raise ValueError('Require 0 <= overlap < chunk_tokens.')
    identity=digest({'files':{name:file_hash(root/name) for name in DATA_FILES},'tokenizer':file_hash(tokenizer_path),
        'chunk_tokens':chunk_tokens,'overlap':overlap,'version':1})
    manifest=output/'manifest.json'
    if manifest.exists():
        old=read_json(manifest)
        if old['fingerprint']!=identity:raise ValueError('Input/cache changed. Use a new PROJECT.')
        if any(old['docs'][i]['gold'] is not None for i in old['splits']['test']):raise ValueError('Unsafe cache contains test targets.')
        if (output/'tokens.i32').stat().st_size!=4*old['total_tokens']:raise ValueError('Incomplete token cache.')
        return str(manifest)
    rows,audit=load_records(root,True)
    label_ids=sorted({y for r in rows if r['split']!='test' for y in r['gold']})
    if not label_ids:raise ValueError('No public training/development labels.')
    label_count=max(label_ids)+1
    if label_count>50000:raise ValueError('Label dimension exceeds 50000. Verify that statutory sections are already mapped to evaluator class IDs.')
    tokenizer=Tokenizer.from_file(str(tokenizer_path));tokenizer.no_padding();tokenizer.no_truncation()
    unk=tokenizer.token_to_id('<unk>');unk=0 if unk is None else unk
    docs=[];total=chunks=empty=0;temporary=output/'tokens.i32.tmp'
    cue=re.compile(r'\b(?:sections?|articles?|acts?|statutes?|codes?|provisions?|ipc|crpc|constitution|rules?)\b',re.I)
    with temporary.open('wb') as handle:
        for index,row in enumerate(rows):
            encoded=tokenizer.encode(row['text'],add_special_tokens=False);ids=encoded.ids;offsets=encoded.offsets
            empty+=int(not ids);ids=ids or [unk];offsets=offsets or [(0,0)]
            np.asarray(ids,dtype='<i4').tofile(handle);n=len(ids);final=max(0,n-chunk_tokens)
            starts=list(range(0,final+1,chunk_tokens-overlap))
            if starts[-1]!=final:starts.append(final)
            windows=[];scores=[]
            for start in starts:
                length=min(chunk_tokens,n-start);windows.append([total+start,length,start/max(1,n-1)])
                left=offsets[start][0];right=offsets[start+length-1][1]
                scores.append(len(cue.findall(row['text'][left:right])))
            order=list(dict.fromkeys([0,len(windows)-1]+sorted(range(len(windows)),key=lambda j:(-scores[j],j))))
            docs.append({'id':row['id'],'source_id':row['id'],'gold':row['gold'],'split':row['split'],
                'token_count':n,'chunks':windows,'evidence_order':order,'feature_start':chunks,'text_sha256':digest(row['text'])})
            total+=n;chunks+=len(windows)
            if (index+1)%2000==0:print('Tokenized cases:',index+1,flush=True)
    temporary.replace(output/'tokens.i32')
    splits={s:[i for i,d in enumerate(docs) if d['split']==s] for s in ('train','dev','test')}
    splits['all']=splits['train']+splits['dev'];splits['every']=list(range(len(docs)))
    support=np.zeros(label_count,dtype=np.int64)
    for i in splits['train']:support[docs[i]['gold']]+=1
    value={'fingerprint':identity,'docs':docs,'splits':splits,'total_tokens':total,'total_chunks':chunks,
        'chunk_tokens':chunk_tokens,'overlap':overlap,'label_ids':label_ids,'label_count':label_count,'train_support':support.tolist(),
        'tokenizer_sha256':file_hash(tokenizer_path),'audit':{**audit,'empty_documents':empty,'all_tokens_retained':True,
        'case_counts':{s:len(v) for s,v in splits.items()},'label_count_including_id_gaps':label_count,
        'labels_without_training_positives':np.flatnonzero(support==0).tolist(),
        'adaptation_view':'First, last, then statute-cue windows; full branch retains all windows.',
        'full_feature_gib_per_512d_parent':chunks*1024*2/1024**3}}
    atomic_json(manifest,value);return str(manifest)

def target_matrix(docs,label_count):
    y=np.zeros((len(docs),label_count),dtype=np.float32)
    for i,doc in enumerate(docs):
        if doc['gold'] is None:raise ValueError('Test targets cannot be used for fitting or selection.')
        y[i,doc['gold']]=1
    return y

class Bags(Dataset):
    def __init__(self,manifest,split,features=None,teacher=None,view='all',adapt_chunks=3):
        self.path=Path(manifest);self.data=read_json(self.path);self.rows=self.data['splits'][split]
        self.tokens=np.memmap(self.path.parent/'tokens.i32',dtype='<i4',mode='r');self.features=None
        if features:
            info=read_json(Path(features)/'manifest.json')
            if not info['complete'] or info['data_fingerprint']!=self.data['fingerprint']:raise ValueError('Feature cache mismatch/incomplete.')
            self.features=np.memmap(Path(features)/'features.f16',dtype='<f2',mode='r',shape=tuple(info['shape']))
        if view not in ('all','evidence'):raise ValueError('Unknown context view.')
        self.teacher,self.view,self.adapt_chunks=teacher,view,adapt_chunks
    def __len__(self):return len(self.rows)
    def __getitem__(self,index):return self.document(self.rows[index])
    def document(self,index):
        doc=self.data['docs'][index];windows=doc['chunks'];keep=list(range(len(windows)))
        if self.view=='evidence':keep=sorted(doc['evidence_order'][:self.adapt_chunks])
        position=[[windows[j][2],1-windows[j][2],windows[j][2]**2,float(j==0),float(j==len(windows)-1),math.log1p(windows[j][1])/8] for j in keep]
        target=None
        if doc['gold'] is not None:
            target=np.zeros(self.data['label_count'],dtype=np.float32);target[doc['gold']]=1
        return {'index':index,'id':doc['id'],'chunks':[np.array(self.tokens[windows[j][0]:windows[j][0]+windows[j][1]],dtype=np.int64) for j in keep] if self.features is None else None,
            'features':torch.from_numpy(np.array(self.features[[doc['feature_start']+j for j in keep]],dtype=np.float32)) if self.features is not None else None,
            'position':position,'target':target,'teacher':self.teacher[doc['id']] if self.teacher is not None else None}

def collate(rows):
    B=len(rows);S=max(len(r['position']) for r in rows);positions=torch.zeros(B,S,6);mask=torch.zeros(B,S,dtype=torch.bool)
    owners=[];chunks=[];features=None
    if rows[0]['features'] is not None:features=torch.zeros(B,S,rows[0]['features'].shape[-1])
    for b,row in enumerate(rows):
        n=len(row['position']);positions[b,:n]=torch.tensor(row['position']);mask[b,:n]=True
        if features is not None:features[b,:n]=row['features']
        else:chunks.extend(row['chunks']);owners.extend(b*S+j for j in range(n))
    targets=None if rows[0]['target'] is None else torch.from_numpy(np.stack([r['target'] for r in rows]))
    return {'chunks':chunks,'owners':owners,'features':features,'position':positions,'mask':mask,'target':targets,
        'teacher':torch.stack([torch.as_tensor(r['teacher']) for r in rows]) if rows[0]['teacher'] is not None else None,'rows':rows}

def probabilities(logits):
    x=np.asarray(logits,dtype=np.float64);return 1/(1+np.exp(-np.clip(x,-40,40)))

def eval_dimension(y):
    active=np.flatnonzero(np.asarray(y).sum(0));return int(active.max()+1) if len(active) else np.asarray(y).shape[1]

def metric(y,probability,threshold=.5):
    y=np.asarray(y,dtype=np.uint8);p=np.asarray(probability);pred=p>=np.asarray(threshold)
    n=eval_dimension(y);y=y[:,:n];pred=pred[:,:n]
    return {'macro_f1':float(f1_score(y,pred,average='macro',zero_division=0)),
        'micro_f1':float(f1_score(y,pred,average='micro',zero_division=0)),
        'subset_accuracy':float(np.all(y==pred,axis=1).mean()),'label_dimension_from_truth':n}

def fit_thresholds(y,p,shrink=20.):
    y=np.asarray(y);p=np.asarray(p);grid=np.array([.03,.05,.075,.1,.15,.2,.25,.3,.35,.4,.45,.5,.6,.7,.8])
    global_t=max(grid,key=lambda t:metric(y,p,t)['macro_f1']);threshold=np.full(y.shape[1],global_t,dtype=float)
    support=y.sum(0)
    for j in range(y.shape[1]):
        if support[j]<3:continue
        pred=p[:,j,None]>=grid[None,:];tp=(pred*y[:,j,None]).sum(0);denom=pred.sum(0)+support[j]
        f1=np.divide(2*tp,denom,out=np.zeros(len(grid)),where=denom>0)
        tied=np.flatnonzero(f1>=f1.max()-1e-12);best=tied[np.argmin(abs(grid[tied]-global_t))]
        strength=support[j]/(support[j]+shrink);threshold[j]=(1-strength)*global_t+strength*grid[best]
    if metric(y,p,threshold)['macro_f1']<metric(y,p,global_t)['macro_f1']:threshold[:]=global_t
    return threshold.tolist()

def predict_labels(probability,threshold,allowed_ids):
    p=np.asarray(probability);binary=p>=np.asarray(threshold);allowed=set(allowed_ids)
    return [[int(j) for j in np.flatnonzero(row) if j in allowed] for row in binary]

def write_submission(ids,preds,path):
    if len(ids)!=len(set(ids)) or len(ids)!=len(preds):raise ValueError('Submission IDs/count mismatch.')
    if any(not isinstance(row,list) or any(type(y) is not int or y<0 for y in row) or len(row)!=len(set(row)) for row in preds):raise ValueError('Predictions require unique nonnegative integer label IDs.')
    rows=[{'id':did,'pred':sorted(p)} for did,p in zip(ids,preds)];raw=json.dumps(rows,ensure_ascii=False).encode()
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True);tmp=path.with_suffix('.tmp')
    with zipfile.ZipFile(tmp,'w',zipfile.ZIP_DEFLATED) as archive:archive.writestr('prediction.json',raw)
    tmp.replace(path);return rows,hashlib.sha256(raw).hexdigest()
'''
(CODE/'lsi_data.py').write_text(MODULE_SOURCE.lstrip('\n'),encoding='utf-8')
print('Wrote','lsi_data.py')

In [ ]:
MODULE_SOURCE=r'''
"""Reusable own-foundation encoder components; no task-adapted parent restoration."""
import gc,math,shutil,types

from pathlib import Path

import numpy as np

import torch

from torch import nn

from torch.nn import functional as F

from torch.utils.checkpoint import checkpoint

from legal_models import LegalLM,ModelConfig,Attention

from lsi_data import atomic_json,atomic_torch,read_json,file_hash,digest

class LoRMA(nn.Module):
    def __init__(self,base,rank,alpha):
        super().__init__();self.base=base;self.scale=alpha/rank;base.requires_grad_(False)
        self.A=nn.Linear(base.out_features,rank,bias=False);self.B=nn.Linear(rank,base.out_features,bias=False)
        nn.init.normal_(self.A.weight,std=.02);nn.init.zeros_(self.B.weight)
    def forward(self,x):
        y=self.base(x);return y+self.scale*self.B(self.A(y))

class Bottleneck(nn.Module):
    def __init__(self,width,bottleneck):
        super().__init__();self.norm=nn.LayerNorm(width);self.down=nn.Linear(width,bottleneck);self.up=nn.Linear(bottleneck,width)
        nn.init.zeros_(self.up.weight);nn.init.zeros_(self.up.bias)
    def forward(self,x):return self.up(F.gelu(self.down(self.norm(x))))

def adapter_hook(module,args,value):
    if not isinstance(value,torch.Tensor):raise ValueError('Adapters require classification mode without generation caches.')
    return value+module.task_adapter(value)

def masked_attention(self,x,memory=None,past=None,position=0,use_cache=False):
    if past is not None or use_cache:raise ValueError('CJPE uses complete classification chunks without generation caches.')
    b,t,_=x.shape;source=x if memory is None else memory;s=source.shape[1]
    q=self.q(x).view(b,t,self.heads,self.hd).transpose(1,2)
    k=self.k(source).view(b,s,self.kv_heads,self.hd).transpose(1,2)
    v=self.v(source).view(b,s,self.kv_heads,self.hd).transpose(1,2)
    if self.rotary is not None:q=self.rotary(q);k=self.rotary(k)
    if self.kv_heads!=self.heads:k=k.repeat_interleave(self.heads//self.kv_heads,1);v=v.repeat_interleave(self.heads//self.kv_heads,1)
    valid=self._cj_valid;causal=self.causal;mask=None
    if self.window or not self._cj_all_valid:
        key=(t,s,self.window,causal,x.device)
        if getattr(self,'_cj_allowed_key',None)!=key:
            qi=torch.arange(t,device=x.device)[:,None];ki=torch.arange(s,device=x.device)[None,:]
            allowed=ki<=qi if causal else torch.ones(t,s,device=x.device,dtype=torch.bool)
            if self.window:allowed&=(qi-ki)<self.window
            self._cj_allowed_key=key;self._cj_allowed=allowed
        allowed=self._cj_allowed
        mask=allowed[None,None]&valid[:,None,None,:s];causal=False
    out=F.scaled_dot_product_attention(q,k,v,attn_mask=mask,is_causal=causal,dropout_p=self.dropout if self.training else 0.)
    return self.out(out.transpose(1,2).contiguous().view(b,t,-1)),None

class JudgmentModel(nn.Module):
    def __init__(self,backbone,cfg):
        super().__init__();self.backbone=backbone;self.cfg=cfg;d=backbone.config.dim;h=cfg.get('head_dim',192)
        self.pool_dim=2*d;method=cfg['method'];backbone.requires_grad_(False)
        if method in ('full','kd','gradual'):backbone.requires_grad_(True)
        elif method=='partial':
            for block in list(backbone.blocks)[-cfg.get('unfrozen_layers',2):]:block.requires_grad_(True)
            backbone.final_norm.requires_grad_(True)
        elif method=='lorma':
            targets={'q','k','v','out','gate','up','down'}
            replacements=[(name,m) for name,m in backbone.named_modules() if isinstance(m,nn.Linear) and name.rsplit('.',1)[-1] in targets]
            for name,m in replacements:
                parent,attr=name.rsplit('.',1);setattr(backbone.get_submodule(parent),attr,LoRMA(m,cfg.get('rank',8),cfg.get('alpha',16.)))
        elif method=='adapter':
            blocks=list(backbone.blocks)+list(backbone.encoder) if hasattr(backbone,'encoder') else list(backbone.blocks)
            for block in blocks:
                block.task_adapter=Bottleneck(d,cfg.get('adapter_dim',48));block.register_forward_hook(adapter_hook)
        elif method=='bitfit':
            biases=[p for name,p in backbone.named_parameters() if 'bias' in name.rsplit('.',1)[-1]]
            if not biases:raise ValueError('This parent has no biases. BitFit uses the hybrid backbone with existing LSTM biases.')
            for p in biases:p.requires_grad_(True)
        else:raise ValueError(f'Unknown method {method}')
        self.attention_modules=[m for m in backbone.modules() if isinstance(m,Attention)]
        for m in self.attention_modules:m.forward=types.MethodType(masked_attention,m);m.dropout=cfg.get('encoder_dropout',.05)
        backbone.config.gradient_checkpointing=False
        for m in backbone.modules():
            if isinstance(m,nn.Dropout):m.p=cfg.get('encoder_dropout',.05)
        if cfg.get('head_only'):backbone.requires_grad_(False)
        self.project=nn.Sequential(nn.LayerNorm(2*d+6),nn.Linear(2*d+6,h),nn.GELU(),nn.Dropout(cfg.get('dropout',.2)))
        self.head_type=cfg.get('head_type','bigru')
        if self.head_type=='transformer':
            layer=nn.TransformerEncoderLayer(h,4,3*h,dropout=cfg.get('dropout',.2),batch_first=True,norm_first=True,activation='gelu')
            self.context=nn.TransformerEncoder(layer,2,enable_nested_tensor=False)
        else:self.context=nn.GRU(h,h//2,num_layers=2,bidirectional=True,batch_first=True,dropout=cfg.get('dropout',.2))
        self.norm=nn.LayerNorm(h);self.attention=nn.Sequential(nn.Linear(h,h//2),nn.Tanh(),nn.Linear(h//2,1,bias=False))
        self.classifier=nn.Sequential(nn.LayerNorm(4*h),nn.Dropout(cfg.get('dropout',.2)),nn.Linear(4*h,2))
    def set_epoch(self,epoch):
        if self.cfg['method']!='gradual' or self.cfg.get('head_only'):return
        count=min(len(self.backbone.blocks),self.cfg.get('unfrozen_layers',2)*2**epoch)
        self.backbone.requires_grad_(False)
        for block in list(self.backbone.blocks)[-count:]:block.requires_grad_(True)
        self.backbone.final_norm.requires_grad_(True)
        if count==len(self.backbone.blocks):self.backbone.requires_grad_(True)
    def _hybrid_block(self,x,valid,block):
        # Re-establish this chunk's mask during backward recomputation as well as forward.
        all_valid=bool(valid.all())
        for module in self.attention_modules:module._cj_valid=valid;module._cj_all_valid=all_valid
        return block(x)
    def _hybrid_tokens(self,ids,valid):
        # Keep the fused cuDNN LSTM out of activation-checkpoint recomputation.
        # This is the same residual LSTM -> Transformer computation as LegalLM.forward.
        x=self.backbone.embedding(ids)
        y,_=self.backbone.rnn(x)
        x=x+self.backbone.rnn_norm(y)
        for block in self.backbone.blocks:
            x=checkpoint(self._hybrid_block,x,valid,block,use_reentrant=False)
        return self.backbone.final_norm(x)
    def pool_tokens(self,ids,valid):
        all_valid=bool(valid.all())
        for module in self.attention_modules:module._cj_valid=valid;module._cj_all_valid=all_valid
        hybrid_checkpoint=(self.backbone.config.architecture=='hybrid' and self.training
            and self.cfg.get('encoder_checkpointing',True)
            and any(p.requires_grad for p in self.backbone.parameters()))
        if hybrid_checkpoint:hidden=self._hybrid_tokens(ids,valid)
        else:hidden=self.backbone(ids,source_ids=ids) if self.backbone.config.architecture=='encoder_decoder' else self.backbone(ids)
        mean=(hidden.float()*valid[:,:,None]).sum(1)/valid.sum(1)[:,None]
        last=hidden[torch.arange(len(ids),device=ids.device),valid.sum(1)-1].float()
        return torch.cat((mean,last),-1)
    def encode_chunks(self,chunks,device):
        values=[];step=self.cfg.get('chunk_batch',8)
        for a in range(0,len(chunks),step):
            local=chunks[a:a+step];T=max(map(len,local));ids=torch.zeros(len(local),T,device=device,dtype=torch.long);valid=torch.zeros_like(ids,dtype=torch.bool)
            for i,row in enumerate(local):ids[i,:len(row)]=torch.as_tensor(row.copy(),device=device);valid[i,:len(row)]=True
            checkpoint_encoder=(self.training and self.cfg.get('encoder_checkpointing',True)
                and any(p.requires_grad for p in self.backbone.parameters()))
            if checkpoint_encoder and self.backbone.config.architecture!='hybrid':
                value=checkpoint(self.pool_tokens,ids,valid,use_reentrant=False)
            else:value=self.pool_tokens(ids,valid)
            values.append(value)
        return torch.cat(values)
    def chunk_features(self,batch):
        if batch['features'] is not None:
            if not self.cfg.get('head_only'):raise ValueError('Frozen feature caches are forbidden after backbone adaptation.')
            return batch['features']
        B,S=batch['mask'].shape;vectors=self.encode_chunks(batch['chunks'],batch['mask'].device)
        indices=torch.tensor(batch['owners'],device=vectors.device);pooled=vectors.new_zeros(B*S,self.pool_dim)
        return pooled.index_copy(0,indices,vectors).reshape(B,S,-1)
    def forward(self,batch):
        x=self.project(torch.cat((self.chunk_features(batch),batch['position']),-1));mask=batch['mask'];B,S,_=x.shape
        if self.head_type=='transformer':x=self.context(x,src_key_padding_mask=~mask)
        else:
            packed=nn.utils.rnn.pack_padded_sequence(x,mask.sum(1).cpu(),batch_first=True,enforce_sorted=False)
            x,_=self.context(packed);x,_=nn.utils.rnn.pad_packed_sequence(x,batch_first=True,total_length=S)
        x=self.norm(x);score=self.attention(x).squeeze(-1).float().masked_fill(~mask,-1e9)
        attention=torch.softmax(score,dim=1);global_pool=(x*attention[:,:,None]).sum(1)
        maximum=x.float().masked_fill(~mask[:,:,None],-1e9).max(1).values
        # Complementary section summaries keep beginning and ending evidence available.
        ordinal=torch.arange(S,device=x.device)[None,:];length=mask.sum(1)[:,None]
        first=mask&(ordinal<torch.ceil(length.float()/4));last=mask&(ordinal>=torch.floor(3*length.float()/4))
        section=lambda m:(x*m[:,:,None]).sum(1)/m.sum(1).clamp_min(1)[:,None]
        vector=torch.cat((global_pool,maximum,section(first),section(last)),-1)
        return {'logits':self.classifier(vector),'attention':attention}

def parameter_report(model):return {'total':sum(p.numel() for p in model.parameters()),'trainable':sum(p.numel() for p in model.parameters() if p.requires_grad),
    'backbone':sum(p.numel() for p in model.backbone.parameters())}
'''
(CODE/'lsi_core.py').write_text(MODULE_SOURCE.lstrip('\n'),encoding='utf-8')
print('Wrote','lsi_core.py')

In [ ]:
MODULE_SOURCE=r'''
"""Phase-1-only LSI: label-wise attention, genuine adaptation, Bernoulli distillation."""
import gc,math,shutil
from pathlib import Path
from dataclasses import asdict
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint
from legal_models import LegalLM,ModelConfig
from lsi_core import JudgmentModel,LoRMA,Bottleneck,parameter_report
from lsi_data import read_json,atomic_json,atomic_torch,digest,file_hash

ARCHITECTURES=('decoder','encoder_decoder','local_gqa','hybrid')

def restore_phase1(root,output,overrides=None,allow_tiny=False):
    root=Path(root);output=Path(output);output.mkdir(parents=True,exist_ok=True);overrides=overrides or {}
    tokenizer=root/'tokenizer.json'
    if not tokenizer.is_file():raise FileNotFoundError(f'Missing {tokenizer}. Attach capstone_phase1 directly.')
    token_hash=file_hash(tokenizer);destination=output/'tokenizer.json'
    if destination.exists() and file_hash(destination)!=token_hash:raise ValueError('Tokenizer changed. Use a new PROJECT.')
    if tokenizer.resolve()!=destination.resolve():shutil.copy2(tokenizer,destination)
    parents={};audit={}
    for architecture in ARCHITECTURES:
        source=Path(overrides.get(architecture,root/'runs'/f'main_{architecture}'/'best_model.pt'))
        if not source.is_file():raise FileNotFoundError(f'Missing Phase-1 parent: {source}')
        sha=file_hash(source);target=output/f'{architecture}.pt'
        if target.exists():
            saved=torch.load(target,map_location='cpu',weights_only=False)
            if saved['provenance']['phase1_sha256']!=sha or saved['provenance']['tokenizer_sha256']!=token_hash:raise ValueError('Parent changed. Use a new PROJECT.')
            count=saved['provenance']['parent_parameters'];del saved
        else:
            saved=torch.load(source,map_location='cpu',weights_only=False)
            if saved.get('training_origin')!='random_initialization_official_phase1_only':raise ValueError('Every experiment must start from your original Phase-1 checkpoint; Phase-2/3/4 task weights are rejected.')
            if saved['tokenizer_sha256']!=token_hash:raise ValueError('Phase-1 checkpoint/tokenizer hash mismatch.')
            config=ModelConfig(**saved['model_config'])
            if config.architecture!=architecture:raise ValueError('Phase-1 architecture mismatch.')
            model=LegalLM(config);model.load_state_dict(saved['model'],strict=True)
            count=sum(p.numel() for p in model.parameters())
            if not allow_tiny and not 50_000_000<=count<=300_000_000:raise ValueError('Phase-1 parent must have 50M–300M parameters.')
            provenance={'phase1_sha256':sha,'tokenizer_sha256':token_hash,'parent_parameters':count,
                'source':str(source),'training_origin':saved['training_origin']}
            atomic_torch(target,{'config':asdict(config),'model':model.state_dict(),'provenance':provenance})
            del model,saved;gc.collect()
        parents[architecture]=str(target);audit[architecture]={'source':str(source),'sha256':sha,'parameters':count}
    atomic_json(output/'input_manifest.json',{'phase1_parents':audit,'tokenizer_sha256':token_hash})
    return parents,str(destination)

def soft_prompt_hook(module,args,value):
    count=module.soft_prompt.shape[0]
    return torch.cat((module.soft_prompt[None].expand(value.shape[0],-1,-1),value[:,count:]),dim=1)

class StatuteModel(JudgmentModel):
    def __init__(self,backbone,cfg):
        setup_cfg={**cfg,'method':'partial','unfrozen_layers':0} if cfg['method']=='prompt' else cfg
        super().__init__(backbone,setup_cfg);self.cfg=cfg
        if cfg['method']=='prompt':
            backbone.requires_grad_(False);count=cfg.get('prompt_tokens',16)
            initial=backbone.embedding.weight[:count].detach().clone()
            backbone.embedding.soft_prompt=nn.Parameter(initial,requires_grad=not cfg.get('head_only',False))
            backbone.embedding.register_forward_hook(soft_prompt_hook)
        h=cfg.get('head_dim',192);labels=cfg['label_count']
        del self.attention,self.classifier
        self.label_queries=nn.Parameter(torch.empty(labels,h));self.label_output=nn.Parameter(torch.empty(labels,h))
        nn.init.normal_(self.label_queries,std=.02);nn.init.normal_(self.label_output,std=.02)
        self.label_bias=nn.Parameter(torch.zeros(labels))
        self.global_classifier=nn.Linear(2*h,labels,bias=False)
        self.chunk_classifier=nn.Linear(h,labels,bias=False)
        self.global_gate=nn.Parameter(torch.tensor(-2.));self.mil_gate=nn.Parameter(torch.tensor(-2.))
        if cfg.get('label_prior') is not None:
            prior=torch.as_tensor(cfg['label_prior']).clamp(.001,.999)
            with torch.no_grad():self.label_bias.copy_(torch.logit(prior))
    def pool_tokens(self,ids,valid):
        if self.cfg['method']!='prompt':return super().pool_tokens(ids,valid)
        count=self.cfg.get('prompt_tokens',16)
        if ids.shape[1]+count>self.backbone.config.max_positions:raise ValueError('Soft prompt plus chunk exceeds the Phase-1 position limit. Reduce chunk_tokens.')
        extended_ids=torch.cat((ids.new_zeros(ids.shape[0],count),ids),1)
        extended_valid=torch.cat((valid.new_ones(valid.shape[0],count),valid),1)
        all_valid=bool(extended_valid.all())
        for module in self.attention_modules:module._cj_valid=extended_valid;module._cj_all_valid=all_valid
        hidden=self.backbone(extended_ids)[:,count:]
        mean=(hidden.float()*valid[:,:,None]).sum(1)/valid.sum(1)[:,None]
        last=hidden[torch.arange(len(ids),device=ids.device),valid.sum(1)-1].float()
        return torch.cat((mean,last),-1)
    def forward(self,batch):
        x=self.project(torch.cat((self.chunk_features(batch),batch['position']),-1));mask=batch['mask'];B,S,_=x.shape
        if self.head_type=='transformer':x=self.context(x,src_key_padding_mask=~mask)
        else:
            packed=nn.utils.rnn.pack_padded_sequence(x,mask.sum(1).cpu(),batch_first=True,enforce_sorted=False)
            x,_=self.context(packed);x,_=nn.utils.rnn.pad_packed_sequence(x,batch_first=True,total_length=S)
        x=self.norm(x)
        scores=torch.einsum('bsh,lh->bsl',x,self.label_queries).float().masked_fill(~mask[:,:,None],-1e9)
        attention=torch.softmax(scores,dim=1)
        pooled=torch.einsum('bsl,bsh->blh',attention,x.float())
        logits=(pooled*self.label_output[None]).sum(-1)+self.label_bias
        mean=(x*mask[:,:,None]).sum(1)/mask.sum(1)[:,None]
        maximum=x.float().masked_fill(~mask[:,:,None],-1e9).max(1).values
        mil=self.chunk_classifier(x).float().masked_fill(~mask[:,:,None],-1e9).max(1).values
        logits=logits+self.global_gate.sigmoid()*self.global_classifier(torch.cat((mean,maximum),-1))+self.mil_gate.sigmoid()*mil
        return {'logits':logits,'attention':attention.mean(-1)}

def build_model(cfg,checkpoint_path=None):
    parent=torch.load(cfg['parent'],map_location='cpu',weights_only=False)
    if parent['provenance'].get('training_origin')!='random_initialization_official_phase1_only':raise ValueError('Invalid Phase-1 lineage.')
    config=ModelConfig(**parent['config']);state=parent['model'];retained=None
    if cfg['method']=='kd':
        n=cfg.get('student_layers',4);original=config.layers
        if config.architecture!='decoder' or not 1<=n<original:raise ValueError('KD student must retain fewer decoder blocks.')
        retained=[round(i*(original-1)/(n-1)) for i in range(n)] if n>1 else [original-1]
        state={k:v for k,v in state.items() if not k.startswith('blocks.')}
        for new,old in enumerate(retained):
            prefix=f'blocks.{old}.';state.update({f'blocks.{new}.'+k[len(prefix):]:v for k,v in parent['model'].items() if k.startswith(prefix)})
        config.layers=n
    backbone=LegalLM(config);backbone.load_state_dict(state,strict=True);model=StatuteModel(backbone,cfg)
    if cfg.get('backbone_init'):
        saved=torch.load(cfg['backbone_init'],map_location='cpu',weights_only=False)
        if saved.get('training_origin')!='own_phase1_phase5_lsi' or saved['config']['architecture']!=cfg['architecture'] or saved['config']['method']!=cfg['method']:
            raise ValueError('Encoder-stage initialization has wrong task/family/method.')
        if saved['provenance']['phase1_sha256']!=parent['provenance']['phase1_sha256']:raise ValueError('Phase-1 lineage changed.')
        learned={k:v for k,v in saved['model'].items() if k.startswith('backbone.')}
        if not learned:raise ValueError('Adapted checkpoint has no encoder updates.')
        _,unexpected=model.load_state_dict(learned,strict=False)
        if unexpected:raise ValueError('Adapted encoder does not match model skeleton.')
    path=checkpoint_path or cfg.get('head_init')
    if path:
        saved=torch.load(path,map_location='cpu',weights_only=False)
        if saved.get('training_origin')!='own_phase1_phase5_lsi' or saved['config']['label_count']!=cfg['label_count']:raise ValueError('Head initialization has wrong task/label dimension.')
        if saved['provenance']['phase1_sha256']!=parent['provenance']['phase1_sha256']:raise ValueError('Head/checkpoint belongs to a different Phase-1 parent.')
        weights=saved['model'] if checkpoint_path else {k:v for k,v in saved['model'].items() if not k.startswith('backbone.')}
        missing,unexpected=model.load_state_dict(weights,strict=False)
        required={k for k,p in model.named_parameters() if p.requires_grad} if checkpoint_path else {k for k in model.state_dict() if not k.startswith('backbone.')}
        if unexpected or set(missing)&required:raise ValueError('Incomplete LSI checkpoint.')
    return model,{**parent['provenance'],'retained_parent_blocks':retained,'backbone_config':asdict(config)}

def classification_loss(logits,batch,cfg,weights):
    if batch['target'] is None:raise ValueError('Test labels cannot enter the training loss.')
    z=logits.float();y=batch['target'].float()
    if cfg.get('loss','asl')=='bce':hard=F.binary_cross_entropy_with_logits(z,y,pos_weight=weights)
    else:
        positive=z.sigmoid();negative=(1-positive+cfg.get('asl_clip',.05)).clamp(max=1)
        log_positive=F.logsigmoid(z);log_negative=negative.clamp_min(1e-8).log()
        focus_positive=(1-positive).pow(cfg.get('gamma_positive',0.))
        focus_negative=(1-negative).pow(cfg.get('gamma_negative',4.))
        hard=-(y*focus_positive*log_positive+(1-y)*focus_negative*log_negative).mean()
    if cfg['method']!='kd':return hard,{'hard':float(hard.detach())}
    if batch['teacher'] is None or batch['teacher'].shape!=z.shape:raise ValueError('KD needs real Phase-5 per-label teacher logits.')
    tau=cfg.get('temperature',2.);alpha=cfg.get('kd_alpha',.5)
    teacher=(batch['teacher'].detach().float()/tau).sigmoid().clamp(1e-6,1-1e-6)
    # Independent Bernoulli KL, including both positive and negative outcomes. Never label softmax.
    kl=(teacher*(teacher.log()-F.logsigmoid(z/tau))+(1-teacher)*((1-teacher).log()-F.logsigmoid(-z/tau))).mean()*tau*tau
    return (1-alpha)*hard+alpha*kl,{'hard':float(hard.detach()),'kd':float(kl.detach())}
'''
(CODE/'lsi_model.py').write_text(MODULE_SOURCE.lstrip('\n'),encoding='utf-8')
print('Wrote','lsi_model.py')

In [ ]:
MODULE_SOURCE=r'''
"""Distributed feature extraction, prediction, resumable selection and fixed-epoch refit."""
import argparse,contextlib,gc,json,math,os,random,time
from pathlib import Path
import numpy as np
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader,Sampler
from lsi_data import Bags,collate,to_device,read_json,atomic_json,atomic_torch,file_hash,digest,metric,probabilities,target_matrix,fit_thresholds
from lsi_model import build_model,parameter_report,classification_loss

def setup():
    rank=int(os.environ.get('RANK',0));world=int(os.environ.get('WORLD_SIZE',1));local=int(os.environ.get('LOCAL_RANK',0))
    device=torch.device('cuda',local) if torch.cuda.is_available() else torch.device('cpu')
    if device.type=='cuda':torch.cuda.set_device(local)
    if world>1 and not dist.is_initialized():
        if device.type=='cuda':dist.init_process_group('nccl',device_id=device)
        else:dist.init_process_group('gloo')
    return rank,world,device
def barrier():
    if dist.is_initialized():dist.barrier()
def amp(device):return torch.autocast('cuda',dtype=torch.float16) if device.type=='cuda' else contextlib.nullcontext()
def signature(cfg):
    identity={'parent':file_hash(cfg['parent']),'student_layers':cfg.get('student_layers') if cfg['method']=='kd' else None,'pool':'mean_last','mask':'inherited','prompt_tokens':cfg.get('prompt_tokens',16) if cfg['method']=='prompt' else None}
    if cfg.get('backbone_init'):identity['adapted_encoder']=file_hash(cfg['backbone_init'])
    return digest(identity)
def rng_state(device):return {'python':random.getstate(),'numpy':np.random.get_state(),'torch':torch.get_rng_state(),
    'cuda':torch.cuda.get_rng_state(device) if device.type=='cuda' else None}
def restore_rng(state,device):
    random.setstate(state['python']);np.random.set_state(state['numpy']);torch.set_rng_state(state['torch'])
    if device.type=='cuda':torch.cuda.set_rng_state(state['cuda'],device)
class LengthSampler(Sampler):
    def __init__(self,dataset,rank,world,seed):self.dataset,self.rank,self.world,self.seed,self.epoch=dataset,rank,world,seed,0
    def set_epoch(self,epoch):self.epoch=epoch
    def __len__(self):return math.ceil(len(self.dataset)/self.world)
    def __iter__(self):
        generator=torch.Generator().manual_seed(self.seed+self.epoch);order=torch.randperm(len(self.dataset),generator=generator).tolist();bucket=[]
        for first in range(0,len(order),256):
            bucket.extend(sorted(order[first:first+256],key=lambda i:len(self.dataset.data['docs'][self.dataset.rows[i]]['chunks'])))
        extra=len(self)*self.world-len(bucket)
        bucket.extend((bucket*math.ceil(extra/max(1,len(bucket))))[:extra])
        return iter(bucket[self.rank::self.world])

@torch.inference_mode()
def infer_local(model,dataset,rank,world,device,batch_size=1,sections='all'):
    model.eval();values={};attention={};indices=list(range(rank,len(dataset),world))
    # Cached feature batches avoid thousands of tiny GPU launches during head selection.
    for first in range(0,len(indices),batch_size):
        rows=[dataset[i] for i in indices[first:first+batch_size]]
        if sections!='all':
            for row in rows:
                n=len(row['position']);keep=list(range(n))
                if sections=='first':keep=keep[:max(1,math.ceil(n/4))]
                elif sections=='last':keep=keep[-max(1,math.ceil(n/4)):]
                else:raise ValueError(sections)
                row['position']=[row['position'][j] for j in keep]
                if row['features'] is not None:row['features']=row['features'][keep]
                else:row['chunks']=[row['chunks'][j] for j in keep]
        batch=to_device(collate(rows),device)
        with amp(device):out=model(batch)
        for i,row in enumerate(rows):
            values[row['id']]=out['logits'][i].float().cpu()
            if len(attention)<8:attention[row['id']]=out['attention'][i,:len(row['position'])].float().cpu().tolist()
    return values,attention
def gather_values(local,world):
    if world==1:return local
    parts=[None]*world;dist.all_gather_object(parts,local);result={}
    for part in parts:
        if set(part)&set(result):raise ValueError('Inference duplicated a case across ranks.')
        result.update(part)
    return result
def feature_cache(args,cfg,rank,world,device):
    root=Path(args.output);root.mkdir(parents=True,exist_ok=True);dataset=Bags(args.data,'every');data=dataset.data
    model,_=build_model(cfg);model.to(device).eval();width=model.pool_dim
    identity=digest({'data':data['fingerprint'],'encoder':signature(cfg),'shape':[data['total_chunks'],width]})
    manifest=root/'manifest.json';old=read_json(manifest) if manifest.exists() else None
    if old and old['identity']!=identity:raise ValueError('Feature cache source changed.')
    if old and old.get('complete'):
        if (root/'features.f16').stat().st_size!=2*data['total_chunks']*width:raise ValueError('Frozen feature file is incomplete.')
        if rank==0:print('Reusing frozen features:',root,flush=True)
        return
    if rank==0 and old is None:
        needed=2*data['total_chunks']*width
        if shutil_free(root)<needed+256*1024**2:raise RuntimeError(f'Need {needed/1024**3:.2f} GiB free for this complete feature cache. No cases are dropped.')
        with (root/'features.f16').open('wb') as handle:handle.truncate(needed)
        with (root/'done.u8').open('wb') as handle:handle.truncate(len(data['docs']))
        atomic_json(manifest,{'identity':identity,'complete':False,'shape':[data['total_chunks'],width],
            'encoder_signature':signature(cfg),'data_fingerprint':data['fingerprint']})
    barrier();matrix=np.memmap(root/'features.f16',dtype='<f2',mode='r+',shape=(data['total_chunks'],width))
    done=np.memmap(root/'done.u8',dtype='u1',mode='r+',shape=(len(data['docs']),));pending=[];started=time.monotonic();processed=0
    def flush():
        matrix.flush()
        if pending:done[pending]=1;done.flush();pending.clear()
    def encode_group(group):
        nonlocal processed
        # Combine windows from several short cases so both GPUs receive useful encoder batches.
        flat=[chunk for _,row in group for chunk in row['chunks']]
        with amp(device):features=model.encode_chunks(flat,device)
        vectors=features.float().cpu().numpy().astype('<f2');cursor=0
        for index,row in group:
            doc=data['docs'][index];n=len(row['chunks']);a=doc['feature_start']
            matrix[a:a+n]=vectors[cursor:cursor+n];cursor+=n;pending.append(index);processed+=1
        if len(pending)>=64:flush()
        if processed and processed//500!=(processed-len(group))//500 and rank==0:
            print(json.dumps({'feature_cases_rank0':processed,'elapsed_minutes':(time.monotonic()-started)/60}),flush=True)
    with torch.inference_mode():
        group=[];count=0
        for index in range(rank,len(data['docs']),world):
            if done[index]:continue
            row=dataset.document(index);group.append((index,row));count+=len(row['chunks'])
            if count>=cfg.get('chunk_batch',16) or len(group)>=16:
                encode_group(group);group=[];count=0
        if group:encode_group(group)
    flush();barrier()
    if rank==0:
        if not np.all(done):raise ValueError('Feature extraction omitted cases.')
        info=read_json(manifest);info.update(complete=True,elapsed_seconds=time.monotonic()-started);atomic_json(manifest,info)
    barrier()
def shutil_free(path):
    import shutil
    return shutil.disk_usage(path).free
def predict(args,cfg,rank,world,device):
    saved=torch.load(args.checkpoint,map_location='cpu',weights_only=False);cfg=saved['config'];root=Path(args.output);root.mkdir(parents=True,exist_ok=True)
    dataset=Bags(args.data,args.split,args.features,view=cfg.get('view','all'),adapt_chunks=cfg.get('adapt_chunks',3))
    if dataset.data['fingerprint']!=saved['data_fingerprint']:raise ValueError('Prediction dataset differs from the checkpoint.')
    if args.features:
        if not cfg.get('head_only'):raise ValueError('An adapted checkpoint must recompute its encoder features.')
        if read_json(Path(args.features)/'manifest.json')['encoder_signature']!=signature(cfg):raise ValueError('Wrong frozen feature encoder.')
    identity=digest({'checkpoint':file_hash(args.checkpoint),'data':dataset.data['fingerprint'],'split':args.split,'sections':args.sections})
    complete=root/'complete.pt'
    if complete.exists():
        old=torch.load(complete,map_location='cpu',weights_only=False)
        if old['identity']!=identity:raise ValueError('Prediction cache source changed.')
        if rank==0:print('Reusing logits:',root,flush=True)
        return
    model,_=build_model(cfg,args.checkpoint);model.to(device)
    local,attention=infer_local(model,dataset,rank,world,device,cfg.get('head_batch',16) if args.features else 1,args.sections)
    values=gather_values(local,world)
    if rank==0:
        expected=[dataset.data['docs'][i]['id'] for i in dataset.rows]
        if set(expected)!=set(values):raise ValueError('Prediction cache ID mismatch.')
        atomic_torch(complete,{'identity':identity,'values':values,'split':args.split,'data_fingerprint':dataset.data['fingerprint'],
            'checkpoint_sha256':file_hash(args.checkpoint),'config':cfg,'parameters':saved['parameters'],'attention_examples_rank0':attention})
    barrier()

def train(args,cfg,rank,world,device):
    root=Path(args.output);root.mkdir(parents=True,exist_ok=True);seed=cfg.get('seed',230437)
    random.seed(seed+rank);np.random.seed(seed+rank);torch.manual_seed(seed+rank)
    if bool(args.features)!=bool(cfg.get('head_only')):raise ValueError('Features are required exactly for frozen-backbone head training.')
    teacher=None;data=read_json(args.data)
    if args.features and read_json(Path(args.features)/'manifest.json')['encoder_signature']!=signature(cfg):raise ValueError('Frozen features came from another parent.')
    if args.teacher:
        cached=torch.load(args.teacher,map_location='cpu',weights_only=False)
        if cached.get('smoke_only') and not args.smoke:raise ValueError('Synthetic preflight teachers are forbidden in real training.')
        if cached['data_fingerprint']!=data['fingerprint'] or cached['split']!=args.split:raise ValueError('Teacher cache has the wrong dataset or training split.')
        teacher=cached['values']
    if cfg['method']=='kd' and teacher is None:raise ValueError('KD requires cached Phase-4 binary teacher logits.')
    dataset=Bags(args.data,args.split,args.features,teacher,view=cfg.get('view','all'),adapt_chunks=cfg.get('adapt_chunks',3))
    if teacher is not None and any(dataset.data['docs'][i]['id'] not in teacher for i in dataset.rows):raise ValueError('Teacher cache omits training cases.')
    dev=Bags(args.data,'dev',args.features,view=cfg.get('view','all'),adapt_chunks=cfg.get('adapt_chunks',3));sampler=LengthSampler(dataset,rank,world,seed)
    generator=torch.Generator();loader=DataLoader(dataset,batch_size=cfg.get('batch_docs',1),sampler=sampler,collate_fn=collate,
        num_workers=0,pin_memory=device.type=='cuda',generator=generator)
    model,provenance=build_model(cfg);model.to(device);report=parameter_report(model)
    if cfg['method']=='kd' and args.teacher:
        if cached['teacher_parameters']<=report['total']:raise ValueError('The teacher must have more parameters than this student.')
    # Gradual unfreezing keeps optimizer membership stable; DDP is rebuilt after each policy change.
    parameters=[(n,p) for n,p in model.named_parameters() if p.requires_grad]
    groups=[]
    for is_encoder in (True,False):
        for decay in (True,False):
            group=[p for n,p in parameters if n.startswith('backbone.')==is_encoder and (p.ndim>=2)==decay]
            if group:groups.append({'params':group,'lr':cfg['encoder_lr'] if is_encoder else cfg['head_lr'],
                'base_lr':cfg['encoder_lr'] if is_encoder else cfg['head_lr'],'weight_decay':cfg.get('weight_decay',.01) if decay else 0.})
    optimizer=torch.optim.AdamW(groups,fused=device.type=='cuda');scaler=torch.amp.GradScaler('cuda',enabled=device.type=='cuda')
    epochs=args.epochs or cfg['epochs'];accum=cfg.get('accumulation',1);planned=math.ceil(len(loader)/accum)*epochs
    support=target_matrix([data['docs'][i] for i in dataset.rows],data['label_count']).sum(0)
    weights=np.sqrt((len(dataset)-support+1)/(support+1)).clip(1,12)
    weights=torch.tensor(weights,dtype=torch.float32,device=device)
    parent_sha=file_hash(cfg['parent'])
    identity=digest({'config':cfg,'data':data['fingerprint'],'parent':parent_sha,'epochs':epochs,'split':args.split,'world':world,
        'teacher':file_hash(args.teacher) if args.teacher else None,'features':signature(cfg) if args.features else None,
        'head_init':file_hash(cfg['head_init']) if cfg.get('head_init') else None,
        'backbone_init':file_hash(cfg['backbone_init']) if cfg.get('backbone_init') else None,'smoke':args.smoke})
    latest=root/'latest.pt';summary_path=root/'summary.json';best=root/'best.pt'
    if summary_path.exists():
        old=read_json(summary_path)
        if old['identity']!=identity:raise ValueError('Run configuration changed. Use a new PROJECT.')
        if old.get('complete') and best.exists():
            if rank==0:print('Already complete:',root,flush=True)
            return
    epoch=cursor=updates=skipped=bad=0;best_f1=-1.;best_epoch=0;history=[];elapsed_prior=0.
    if latest.exists():
        old=torch.load(latest,map_location='cpu',weights_only=False)
        if old['identity']!=identity:raise ValueError('Resume identity mismatch.')
        model.load_state_dict(old['model'],strict=not old['delta_only']);optimizer.load_state_dict(old['optimizer']);scaler.load_state_dict(old['scaler'])
        epoch,cursor,updates,skipped,bad=[old[k] for k in ('epoch','cursor','updates','skipped','bad')]
        best_f1,best_epoch,history,elapsed_prior=old['best_f1'],old['best_epoch'],old['history'],old['elapsed']
        restore_rng(old['rng'][rank],device)
    started=time.monotonic();trainable_snapshot={n for n,p in model.named_parameters() if p.requires_grad}
    delta_only=cfg.get('head_only') or cfg['method'] not in ('full','kd','gradual')
    def save(is_best=False,complete=False):
        states=[None]*world
        if world>1:dist.all_gather_object(states,rng_state(device))
        else:states[0]=rng_state(device)
        if rank==0:
            state=model.state_dict()
            if delta_only:state={k:v for k,v in state.items() if k in trainable_snapshot}
            light={'model':state,'delta_only':delta_only,'config':cfg,'provenance':provenance,'parameters':report,
                'data_fingerprint':data['fingerprint'],'training_origin':'own_phase1_phase5_lsi',
                'parent_sha256':parent_sha,
                'stage':args.split,'epoch':epoch,'updates':updates}
            if is_best:atomic_torch(best,light)
            checkpoint={**light,'identity':identity,'optimizer':optimizer.state_dict(),'scaler':scaler.state_dict(),'rng':states,
                'cursor':cursor,'skipped':skipped,'bad':bad,'best_f1':best_f1,'best_epoch':best_epoch,'history':history,'elapsed':elapsed_prior+time.monotonic()-started}
            atomic_torch(latest,checkpoint);atomic_json(summary_path,{'identity':identity,'complete':complete,'best_epoch':best_epoch,
                'best_f1':best_f1,'epoch':epoch,'updates':updates,'amp_skipped_updates':skipped,'parameters':report,
                'elapsed_seconds':checkpoint['elapsed'],'history':history,'stop_reason':'smoke' if args.smoke else 'epochs_or_patience'})
        barrier()
    if rank==0:print(json.dumps({'experiment':cfg['name'],'stage':args.split,'head_only':cfg.get('head_only',False),
        'cases':len(dataset),'parameters':report,'effective_batch':cfg.get('batch_docs',1)*world*accum}),flush=True)
    while epoch<epochs:
        model.set_epoch(epoch);sampler.set_epoch(epoch);generator.manual_seed(seed+epoch)
        wrapped=DDP(model,device_ids=[device.index] if device.type=='cuda' else None,broadcast_buffers=False) if world>1 else model
        iterator=iter(loader)
        for _ in range(cursor):next(iterator)
        model.train();loss_sum=loss_count=0
        while cursor<len(loader):
            micro=[next(iterator) for _ in range(min(accum,len(loader)-cursor))];sample_count=sum(len(b['rows']) for b in micro);optimizer.zero_grad(set_to_none=True)
            ratio=(updates+1)/max(1,planned);warm=cfg.get('warmup_ratio',.08)
            factor=min(1.,ratio/max(warm,1e-6)) if ratio<warm else .1+.9*.5*(1+math.cos(math.pi*min(1.,(ratio-warm)/max(1-warm,1e-6))))
            for group in optimizer.param_groups:group['lr']=group['base_lr']*factor
            for index,batch in enumerate(micro):
                batch=to_device(batch,device);sync=wrapped.no_sync() if world>1 and index<len(micro)-1 else contextlib.nullcontext()
                with sync:
                    with amp(device):result=wrapped(batch);loss,components=classification_loss(result['logits'],batch,cfg,weights)
                    if not bool(torch.isfinite(loss)):raise FloatingPointError('Nonfinite loss; recovery states were retained.')
                    scaler.scale(loss*len(batch['rows'])/sample_count).backward()
                loss_sum+=float(loss.detach());loss_count+=1
            scaler.unscale_(optimizer);gradient=torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],1.)
            scale=scaler.get_scale();scaler.step(optimizer);scaler.update();skipped+=int(scaler.get_scale()<scale);updates+=int(scaler.get_scale()>=scale);cursor+=len(micro)
            if rank==0 and updates and updates%cfg.get('log_every',50)==0:
                print(json.dumps({'epoch':epoch+1,'batch':cursor,'updates':updates,'loss':loss_sum/max(1,loss_count),
                    'gradient_norm':float(gradient),'elapsed_hours':(elapsed_prior+time.monotonic()-started)/3600,'amp_skipped_updates':skipped}),flush=True)
                loss_sum=loss_count=0
            if cursor<len(loader) and updates and updates%cfg.get('checkpoint_every',100)==0:save()
            if args.smoke:
                epoch+=1;cursor=0;best_epoch=epoch;best_f1=0.;save(True,True)
                if rank==0:print('PASS actual distributed update:',cfg['name'],flush=True)
                return
        epoch+=1;cursor=0
        if args.split=='train':
            values,_=infer_local(model,dev,rank,world,device,cfg.get('head_batch',16) if args.features else 1)
            values=gather_values(values,world);ordered=[data['docs'][i] for i in data['splits']['dev']]
            truth=target_matrix(ordered,data['label_count']);p=probabilities(torch.stack([values[d['id']] for d in ordered]).numpy())
            p[:,np.asarray(data['train_support'])==0]=0
            result=metric(truth,p,fit_thresholds(truth,p))
            score=result['macro_f1'];history.append({'epoch':epoch,**result})
            improved=score>best_f1+cfg.get('min_delta',0.)
            if improved:best_f1=score;best_epoch=epoch;bad=0
            else:bad+=1
            if rank==0:print(json.dumps({'epoch':epoch,'dev_macro_f1_calibrated':score,'best_epoch':best_epoch}),flush=True)
        else:improved=True;best_epoch=epoch;best_f1=0.
        save(improved,epoch>=epochs or (args.split=='train' and bad>=cfg.get('patience',2)))
        del wrapped;gc.collect()
        if args.split=='train' and bad>=cfg.get('patience',2):break
    save(False,True)

def main():
    p=argparse.ArgumentParser();p.add_argument('--action',choices=['features','train','predict'],required=True)
    p.add_argument('--config');p.add_argument('--checkpoint');p.add_argument('--data',required=True);p.add_argument('--output',required=True)
    p.add_argument('--split',default='train',choices=['train','dev','test','all','every']);p.add_argument('--features');p.add_argument('--teacher')
    p.add_argument('--epochs',type=int);p.add_argument('--smoke',action='store_true');p.add_argument('--sections',default='all',choices=['all','first','last'])
    args=p.parse_args();rank,world,device=setup()
    cfg=read_json(args.config) if args.config else torch.load(args.checkpoint,map_location='cpu',weights_only=False)['config']
    try:
        if args.action=='features':feature_cache(args,cfg,rank,world,device)
        elif args.action=='predict':predict(args,cfg,rank,world,device)
        else:train(args,cfg,rank,world,device)
    finally:
        if dist.is_initialized():dist.destroy_process_group()
if __name__=='__main__':main()
'''
(CODE/'lsi_run.py').write_text(MODULE_SOURCE.lstrip('\n'),encoding='utf-8')
print('Wrote','lsi_run.py')

In [ ]:
MODULE_SOURCE=r'''
"""Seven Phase-5 methods: shared full-document banks plus adapted evidence and dev-only fusion."""
import contextlib,gc,json,math,os,shutil,socket,subprocess,sys,time,zipfile
from pathlib import Path
import numpy as np
import torch
from lsi_data import *
from lsi_model import restore_phase1,build_model,parameter_report,classification_loss
def obtain_data(project,local_root=None,revision=None):
    destination=Path(project)/'data';destination.mkdir(parents=True,exist_ok=True);provenance=destination/'source.json'
    if local_root:
        source=Path(local_root)
        if (source/'Phase_5_LSI').is_dir():source/='Phase_5_LSI'
        missing=[name for name in DATA_FILES if not (source/name).is_file()]
        if missing:raise FileNotFoundError('Missing official Phase-5 files: '+', '.join(missing))
        origin={'kind':'attached_official_phase5','source':str(source)}
        for name in DATA_FILES:
            target=destination/name;target.parent.mkdir(parents=True,exist_ok=True)
            if target.resolve()!=(source/name).resolve():shutil.copy2(source/name,target)
    else:
        old=read_json(provenance) if provenance.exists() else None
        if old and old.get('kind')=='official_huggingface' and old.get('sha256') and (revision is None or revision==old.get('revision')):
            if all((destination/name).is_file() for name in DATA_FILES):
                hashes={name:file_hash(destination/name) for name in DATA_FILES}
                if hashes!=old['sha256']:raise ValueError('Cached official data changed; restore it or use a new PROJECT.')
                return str(destination)
        from huggingface_hub import HfApi,hf_hub_download
        token=os.environ.get('HF_TOKEN')
        if not token:
            try:
                from kaggle_secrets import UserSecretsClient
                token=UserSecretsClient().get_secret('HF_TOKEN')
            except Exception:token=None
        repo='Exploration-Lab/CS-781-Capstone';commit=old['revision'] if old and old.get('kind')=='official_huggingface' else revision
        try:
            commit=HfApi(token=token).dataset_info(repo,revision=commit).sha
            origin={'kind':'official_huggingface','repo':repo,'revision':commit};atomic_json(provenance,origin)
            for name in DATA_FILES:
                target=destination/name;target.parent.mkdir(parents=True,exist_ok=True)
                if not target.exists():
                    source=hf_hub_download(repo,'Phase_5_LSI/'+name,repo_type='dataset',revision=commit,token=token)
                    shutil.copy2(source,target)
        except Exception as error:raise RuntimeError('Attach Phase_5_LSI, or accept the official dataset terms and enable the HF_TOKEN Kaggle secret. Never paste API tokens into code.') from error
    origin['sha256']={name:file_hash(destination/name) for name in DATA_FILES};atomic_json(provenance,origin)
    return str(destination)

def launch(action,args,project,gpus=2,log=None):
    script=Path(project)/'code/lsi_run.py'
    if gpus==1:cmd=[sys.executable,'-u',str(script),'--action',action,*map(str,args)]
    else:
        with socket.socket() as sock:sock.bind(('127.0.0.1',0));port=sock.getsockname()[1]
        cmd=[sys.executable,'-m','torch.distributed.run','--nnodes=1','--master_addr=127.0.0.1',f'--master_port={port}',
            f'--nproc_per_node={gpus}',str(script),'--action',action,*map(str,args)]
    env=dict(os.environ);env['PYTHONPATH']=str(Path(project)/'code')+os.pathsep+env.get('PYTHONPATH','')
    env['TOKENIZERS_PARALLELISM']='false';env['OMP_NUM_THREADS']='1'
    handle=Path(log).open('a',encoding='utf-8') if log else None
    try:
        process=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=env)
        for line in process.stdout:
            print(line,end='',flush=True)
            if handle:handle.write(line);handle.flush()
        if process.wait()!=0:raise RuntimeError(f'{action} failed. Inspect {log or "cell output"}. Earlier candidates and resume states remain available.')
    finally:
        if handle:handle.close()

def benchmark(rows,data_path,device=None):
    device=device or torch.device('cuda',0);data=read_json(data_path);dataset=Bags(data_path,'train');measurements=[]
    if device.type=='cuda':torch.cuda.set_device(device)
    for cfg in rows:
        dataset=Bags(data_path,'train',view=cfg.get('view','all'),adapt_chunks=cfg.get('adapt_chunks',3))
        torch.manual_seed(cfg['seed']);model,_=build_model(cfg);model.to(device).train();model.set_epoch(0)
        row=dataset[0];row['chunks']=row['chunks'][:4];row['position']=row['position'][:4]
        batch=to_device(collate([row]),device)
        if cfg['method']=='kd':batch['teacher']=torch.zeros(1,cfg['label_count'],device=device)
        optimizer=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=cfg['head_lr'],fused=device.type=='cuda')
        scaler=torch.amp.GradScaler('cuda',enabled=device.type=='cuda');times=[];peak=0
        for iteration in range(3):
            if device.type=='cuda':torch.cuda.synchronize();torch.cuda.reset_peak_memory_stats()
            start=time.monotonic();optimizer.zero_grad(set_to_none=True)
            with torch.autocast('cuda',dtype=torch.float16) if device.type=='cuda' else contextlib.nullcontext():
                result=model(batch);loss,_=classification_loss(result['logits'],batch,cfg,torch.ones(2,device=device))
            scaler.scale(loss).backward();scaler.step(optimizer);scaler.update()
            if device.type=='cuda':torch.cuda.synchronize();peak=max(peak,torch.cuda.max_memory_reserved()/1024**3)
            if iteration:times.append(time.monotonic()-start)
        n=len(row['chunks']);seconds=sum(times)/len(times);total_chunks=sum(len(data['docs'][i]['chunks']) if cfg.get('view','all')=='all' else min(len(data['docs'][i]['chunks']),cfg.get('adapt_chunks',3)) for i in data['splits']['train'])
        result={'name':cfg['name'],'timed_chunks':n,'seconds_per_sample':seconds,'seconds_per_chunk':seconds/n,
            'peak_reserved_gib':peak,'parameters':parameter_report(model),'rough_adaptation_hours':total_chunks/n*seconds*cfg['epochs']/2/3600,
            'forecast_excludes':'feature extraction, heads, validation, KD caches, DDP, loading and I/O; variable-length bags differ'}
        measurements.append(result);print(json.dumps(result),flush=True)
        del model,optimizer,scaler,batch,result,loss;gc.collect()
        if device.type=='cuda':torch.cuda.empty_cache()
    return measurements

def smoke_all(project,rows,data_path,settings):
    project=Path(project);data=read_json(data_path);root=project/'preflight';root.mkdir(parents=True,exist_ok=True)
    teacher=root/'synthetic_teacher_for_smoke_only.pt'
    atomic_torch(teacher,{'data_fingerprint':data['fingerprint'],'split':'train','teacher_parameters':10**12,'smoke_only':True,
        'values':{data['docs'][i]['id']:torch.zeros(data['label_count']) for i in data['splits']['train']}})
    for cfg in rows:
        current={**cfg,'batch_docs':1,'accumulation':1,'epochs':1,'checkpoint_every':10**9};path=root/f"{cfg['name']}.json";atomic_json(path,current)
        destination=root/cfg['name'];args=['--config',path,'--data',data_path,'--output',destination,'--smoke']
        if cfg['method']=='kd':args+=['--teacher',teacher]
        launch('train',args,project,settings['gpus'],root/f"{cfg['name']}.log")
        if not read_json(destination/'summary.json')['complete']:raise ValueError('Incomplete GPU smoke test.')
        shutil.rmtree(destination)
    teacher.unlink();atomic_json(root/'passed.json',{'experiments':[r['name'] for r in rows],'data_fingerprint':data['fingerprint'],'world_size':settings['gpus']})

def bundle_working(working,destination):
    working=Path(working).resolve();destination=Path(destination).resolve();temporary=destination.with_suffix('.tmp')
    with zipfile.ZipFile(temporary,'w',zipfile.ZIP_DEFLATED,compresslevel=1) as archive:
        for source in working.rglob('*'):
            if source.is_file() and source.resolve() not in (destination,temporary):archive.write(source,source.relative_to(working))
    temporary.replace(destination);return str(destination)

def experiments(parents,settings,data):
    support=np.asarray(data['train_support']);prior=((support+.5)/(len(data['splits']['train'])+1)).tolist()
    common={'head_dim':settings['head_dim'],'dropout':.2,'encoder_dropout':.05,'chunk_batch':settings['chunk_batch'],
        'encoder_checkpointing':True,'head_only':False,'batch_docs':settings['batch_docs'],'head_batch':settings['head_batch'],
        'accumulation':settings['accumulation'],'encoder_lr':1e-5,'head_lr':3e-4,'weight_decay':.01,
        'epochs':settings['adapt_epochs'],'patience':settings['adapt_patience'],'warmup_ratio':.08,
        'rank':8,'alpha':16.,'adapter_dim':48,'unfrozen_layers':2,'seed':230437,'checkpoint_every':100,'log_every':50,
        'temperature':2.,'kd_alpha':.5,'view':'evidence','adapt_chunks':settings['adapt_chunks'],'prompt_tokens':16,
        'label_count':data['label_count'],'label_prior':prior,'loss':'asl','gamma_negative':4.,'gamma_positive':0.,'asl_clip':.05}
    specs=[('01_decoder_full','decoder','full','bigru'),('02_encoder_decoder_partial','encoder_decoder','partial','transformer'),
        ('03_local_gqa_lorma','local_gqa','lorma','bigru'),('04_hybrid_adapter','hybrid','adapter','bigru'),
        ('05_decoder_prompt','decoder','prompt','bigru'),('06_encoder_decoder_gradual','encoder_decoder','gradual','bigru'),
        ('07_decoder_student_kd','decoder','kd','bigru')]
    rows=[]
    for name,architecture,method,head in specs:
        cfg={**common,'name':name,'architecture':architecture,'method':method,'head_type':head,'parent':parents[architecture]}
        if method in ('lorma','adapter'):cfg['encoder_lr']=1e-4
        if method=='prompt':cfg['encoder_lr']=5e-3
        if method=='kd':cfg['student_layers']=settings['student_layers']
        rows.append(cfg)
    return rows

def frozen_key(cfg):
    return digest({'parent':file_hash(cfg['parent']),'student_layers':cfg.get('student_layers') if cfg['method']=='kd' else None})[:16]

def warm_config(cfg,settings):
    return {**cfg,'name':'shared_warm_'+frozen_key(cfg)+'_'+cfg['head_type'],
        'method':'kd' if cfg['method']=='kd' else 'full','head_only':True,'view':'all','encoder_lr':1e-5,
        'epochs':settings['head_epochs'],'patience':settings['head_patience'],'batch_docs':settings['head_batch'],'accumulation':1}

def _predict(project,checkpoint,data_path,split,output,settings,features=None):
    args=['--checkpoint',checkpoint,'--data',data_path,'--split',split,'--output',output]
    if features:args+=['--features',features]
    launch('predict',args,project,1 if features else settings['gpus'],Path(output).parent/(Path(output).name+'.log'))
    return torch.load(Path(output)/'complete.pt',map_location='cpu',weights_only=False)

def _array(raw,data,split):
    return probabilities(torch.stack([raw['values'][data['docs'][i]['id']] for i in data['splits'][split]]).numpy())

def lexical_branch(project,data_root,data_path,max_features=80000):
    import joblib
    from joblib import Parallel,delayed
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import SGDClassifier
    root=Path(project)/'lexical';root.mkdir(parents=True,exist_ok=True);data=read_json(data_path)
    identity=digest({'data':data['fingerprint'],'kind':'train_only_tfidf_sgd_ovr','max_features':max_features,'version':1})
    output=root/'probabilities.pt'
    if output.exists():
        old=torch.load(output,map_location='cpu',weights_only=False)
        if old['identity']!=identity:raise ValueError('Lexical setup changed; use a new PROJECT.')
        return old
    rows=load_records(data_root);byid={r['id']:r for r in rows};texts=[byid[d['id']]['text'] for d in data['docs']]
    train=data['splits']['train'];start=time.monotonic()
    vectorizer=TfidfVectorizer(ngram_range=(1,2),min_df=1,max_features=max_features,sublinear_tf=True,dtype=np.float32,token_pattern=r'(?u)\b\w+\b')
    vectorizer.fit([texts[i] for i in train]);X=vectorizer.transform(texts);y=target_matrix([data['docs'][i] for i in train],data['label_count'])
    def fit_one(j):
        target=y[:,j].astype(int)
        if target.min()==target.max():return None,np.full(X.shape[0],float(target[0]),dtype=np.float32)
        clf=SGDClassifier(loss='log_loss',alpha=1e-5,max_iter=20,tol=1e-3,class_weight='balanced',average=True,random_state=230437)
        clf.fit(X[train],target);return clf,clf.predict_proba(X)[:,1].astype(np.float32)
    fitted=Parallel(n_jobs=2,prefer='threads')(delayed(fit_one)(j) for j in range(data['label_count']))
    matrix=np.stack([v[1] for v in fitted],axis=1);values={doc['id']:matrix[i] for i,doc in enumerate(data['docs'])}
    result={'identity':identity,'values':values,'data_fingerprint':data['fingerprint'],'fit_split':'train_only',
        'elapsed_seconds':time.monotonic()-start,'label_count':data['label_count']}
    joblib.dump({'vectorizer':vectorizer,'classifiers':[v[0] for v in fitted],'constants':y.mean(0),'identity':identity},root/'model.joblib',compress=3)
    atomic_torch(output,result);print('Train-only lexical branch seconds:',round(result['elapsed_seconds'],1),flush=True);return result

def _fuse(warm,adapted,ids,lexical,calibration):
    alpha=calibration['adapt_weight'];p=(1-alpha)*np.asarray(warm)+alpha*np.asarray(adapted)
    weight=calibration.get('lexical_weight',0.)
    if weight:p=(1-weight)*p+weight*np.asarray([lexical['values'][did] for did in ids])
    return p

def calibrate(y,warm,adapted,ids,lexical):
    best=None
    for alpha in (.25,.5,.75,1.):
        for weight in ((0.,.2,.4) if lexical else (0.,)):
            cal={'adapt_weight':alpha,'lexical_weight':weight};p=_fuse(warm,adapted,ids,lexical,cal)
            threshold=fit_thresholds(y,p);scores=metric(y,p,threshold)
            if best is None or scores['macro_f1']>best[0]['macro_f1']+1e-10:best=(scores,{**cal,'thresholds':threshold})
    return best

def _teacher_cache(project,item,data_path,root,settings,lexical):
    root=Path(root);root.mkdir(parents=True,exist_ok=True);data=read_json(data_path);target=root/'teacher.pt'
    identity=digest({'warm':file_hash(item['warm_checkpoint']),'adapted':file_hash(item['adapted_checkpoint']),
        'calibration':item['calibration'],'data':data['fingerprint'],'lexical':lexical['identity'] if lexical else None})
    if target.exists():
        old=torch.load(target,map_location='cpu',weights_only=False)
        if old['identity']!=identity:raise ValueError('Teacher changed. Use a new PROJECT.')
        return target
    warm=_predict(project,item['warm_checkpoint'],data_path,'train',root/'warm',settings,item['features'])
    adapted=_predict(project,item['adapted_checkpoint'],data_path,'train',root/'adapted',settings)
    ids=[data['docs'][i]['id'] for i in data['splits']['train']]
    p=_fuse(_array(warm,data,'train'),_array(adapted,data,'train'),ids,lexical,item['calibration']);p=np.clip(p,1e-6,1-1e-6)
    values={did:torch.tensor(np.log(q/(1-q)),dtype=torch.float32) for did,q in zip(ids,p)}
    atomic_torch(target,{'identity':identity,'data_fingerprint':data['fingerprint'],'split':'train','values':values,
        'teacher_parameters':item['parameters']['total'],'teacher_name':item['name'],'teacher_kind':'phase5_independent_bernoulli_logits',
        'label_count':data['label_count']})
    return target

def run_all(project,rows,data_path,settings,lexical=None):
    project=Path(project);data=read_json(data_path);catalog=[]
    dev=[data['docs'][i] for i in data['splits']['dev']];y=target_matrix(dev,data['label_count'])
    dev_ids=[r['id'] for r in dev];test_ids=[data['docs'][i]['id'] for i in data['splits']['test']]
    allowed=np.flatnonzero(np.asarray(data['train_support'])>0).tolist();dev_allowed=[j for j in allowed if j<eval_dimension(y)]
    atomic_json(project/'dev_truth.json',[{'id':r['id'],'gold':r['gold']} for r in dev])
    for original in rows:
        cfg=dict(original);root=project/'runs'/cfg['name'];root.mkdir(parents=True,exist_ok=True);teacher=None
        if cfg['method']=='kd':
            strongest=max(catalog,key=lambda item:item['dev_metrics']['macro_f1'])
            teacher=_teacher_cache(project,strongest,data_path,root/'teacher_train',settings,lexical);cfg['teacher_name']=strongest['name']
        feature_root=project/'features'/frozen_key(cfg);feature_root.mkdir(parents=True,exist_ok=True)
        warm=warm_config(cfg,settings);feature_cfg=feature_root/'encoder_config.json';atomic_json(feature_cfg,warm)
        launch('features',['--config',feature_cfg,'--data',data_path,'--output',feature_root],project,settings['gpus'],feature_root/'features.log')
        head_candidates=[]
        for loss_kind in settings.get('head_losses',('asl','bce')):
            candidate={**warm,'name':warm['name']+'_'+loss_kind,'loss':loss_kind}
            warm_path=feature_root/f"warm_{cfg['head_type']}_{loss_kind}.json";atomic_json(warm_path,candidate)
            candidate_root=feature_root/f"head_{cfg['head_type']}_{loss_kind}"
            args=['--config',warm_path,'--data',data_path,'--split','train','--output',candidate_root,'--features',feature_root]
            if teacher:args+=['--teacher',teacher]
            launch('train',args,project,1,candidate_root.with_suffix('.log'))
            summary=read_json(candidate_root/'summary.json');head_candidates.append((summary['best_f1'],candidate_root,candidate))
        _,warm_root,warm=max(head_candidates,key=lambda value:value[0]);warm_checkpoint=warm_root/'best.pt'
        cfg['loss']=warm['loss']
        cfg['head_init']=str(warm_checkpoint);cfg_path=root/'config.json';atomic_json(cfg_path,cfg)
        args=['--config',cfg_path,'--data',data_path,'--split','train','--output',root/'adaptation']
        if teacher:args+=['--teacher',teacher]
        launch('train',args,project,settings['gpus'],root/'adaptation.log');adapted_checkpoint=root/'adaptation/best.pt'
        warm_dev=_predict(project,warm_checkpoint,data_path,'dev',root/'warm_dev',settings,feature_root)
        adapted_dev=_predict(project,adapted_checkpoint,data_path,'dev',root/'adapted_dev',settings)
        wp,ap=_array(warm_dev,data,'dev'),_array(adapted_dev,data,'dev')
        wp[:,np.asarray(data['train_support'])==0]=0;ap[:,np.asarray(data['train_support'])==0]=0
        scores,cal=calibrate(y,wp,ap,dev_ids,lexical)
        dev_probability=_fuse(wp,ap,dev_ids,lexical,cal)
        dev_preds=predict_labels(dev_probability,cal['thresholds'],dev_allowed)
        atomic_json(root/'dev_prediction.json',[{'id':did,'pred':p} for did,p in zip(dev_ids,dev_preds)])
        atomic_json(root/'calibration.json',cal)
        warm_test=_predict(project,warm_checkpoint,data_path,'test',root/'warm_test',settings,feature_root)
        adapted_test=_predict(project,adapted_checkpoint,data_path,'test',root/'adapted_test',settings)
        p=_fuse(_array(warm_test,data,'test'),_array(adapted_test,data,'test'),test_ids,lexical,cal)
        pred=predict_labels(p,cal['thresholds'],allowed);submission=project/'submissions'/f"{cfg['name']}.zip"
        prediction,sha=write_submission(test_ids,pred,submission);atomic_json(root/'prediction.json',prediction)
        summary=read_json(adapted_checkpoint.parent/'summary.json');warm_summary=read_json(warm_root/'summary.json')
        item={'name':cfg['name'],'architecture':cfg['architecture'],'method':cfg['method'],'dev_metrics':scores,'calibration':cal,
            'warm_checkpoint':str(warm_checkpoint),'adapted_checkpoint':str(adapted_checkpoint),'features':str(feature_root),
            'parameters':summary['parameters'],'warm_parameters':warm_summary['parameters'],'teacher_name':cfg.get('teacher_name'),
            'submission':str(submission),'prediction_sha256':sha,'dev_used_for_selection':True,
            'head_epochs':warm_summary['best_epoch'],'adapt_epochs':summary['best_epoch'],
            'full_document_branch':cal['adapt_weight']<1.,'all_tokens_encoded_in_frozen_bank':True,'adapted_view':cfg['view'],'adapt_chunks':cfg['adapt_chunks'],
            'selected_loss':cfg['loss'],'warm_head_candidates':[{'loss':r[2]['loss'],'dev_macro_f1':r[0]} for r in head_candidates]}
        catalog.append(item);atomic_json(project/'catalog.json',catalog)
        print('READY:',submission,'dev Macro-F1:',round(scores['macro_f1'],5),'fusion:',cal['adapt_weight'],cal['lexical_weight'],flush=True)
    if len(catalog)!=7 or len({r['architecture'] for r in catalog})!=4:raise ValueError('Experiment coverage incomplete.')
    distinct=len({r['prediction_sha256'] for r in catalog})
    print('Distinct base prediction sets:',distinct,'; general course minimum:',6,flush=True)
    if distinct<6:print('The global distinct-submission requirement is not yet met. Inspect training and dev errors, then improve genuinely different experiments. Completed ZIPs are retained; predictions are not changed artificially.',flush=True)
    best_by_family=[max([r for r in catalog if r['architecture']==a],key=lambda r:r['dev_metrics']['macro_f1']) for a in ('decoder','encoder_decoder','local_gqa','hybrid')]
    atomic_json(project/'best_four_architectures.json',best_by_family)
    required={r['name']:r for r in best_by_family}
    for r in catalog:
        if r['method'] in ('kd','lorma'):required[r['name']]=r
    atomic_json(project/'required_submission_plan.json',{'all_seven':[r['submission'] for r in catalog],
        'architecture_lorma_kd_coverage_subset':[r['submission'] for r in required.values()],
        'global_minimum_distinct_submissions':6,'distinct_base_prediction_sets':len({r['prediction_sha256'] for r in catalog}),
        'global_count_met_by_base_predictions':len({r['prediction_sha256'] for r in catalog})>=6,
        'note':'Submit all seven candidates and verify at least six distinct prediction sets under the general instructions. The family/method subset is coverage analysis, not the overall submission-count requirement. Include KD even when another decoder is stronger. The optional ensemble is not a new adaptation method.'})
    return catalog

def bonus_ensemble(project,data_path,lexical=None):
    project=Path(project);data=read_json(data_path);catalog=read_json(project/'catalog.json')
    ranked=sorted(catalog,key=lambda r:r['dev_metrics']['macro_f1'],reverse=True);families=read_json(project/'best_four_architectures.json')
    dev_ids=[data['docs'][i]['id'] for i in data['splits']['dev']];test_ids=[data['docs'][i]['id'] for i in data['splits']['test']]
    y=target_matrix([data['docs'][i] for i in data['splits']['dev']],data['label_count']);allowed=np.flatnonzero(np.asarray(data['train_support'])>0).tolist()
    base={'name':ranked[0]['name'],'submission':ranked[0]['submission'],'dev_metrics':ranked[0]['dev_metrics'],'kind':'single_adaptation'}
    best=base['dev_metrics']['macro_f1'];winner=None
    for members in (ranked[:3],families):
        devs=[];tests=[]
        for item in members:
            root=project/'runs'/item['name'];cal=item['calibration']
            arrays={}
            for split,ids in (('dev',dev_ids),('test',test_ids)):
                warm=torch.load(root/f'warm_{split}/complete.pt',map_location='cpu',weights_only=False)
                adapted=torch.load(root/f'adapted_{split}/complete.pt',map_location='cpu',weights_only=False)
                raw=_fuse(_array(warm,data,split),_array(adapted,data,split),ids,lexical,cal)
                threshold=np.asarray(cal['thresholds']).clip(1e-5,1-1e-5);raw=raw.clip(1e-5,1-1e-5)
                arrays[split]=probabilities(np.log(raw/(1-raw))-np.log(threshold/(1-threshold)))
            devs.append(arrays['dev']);tests.append(arrays['test'])
        p=np.mean(devs,0);p[:,np.asarray(data['train_support'])==0]=0
        thresholds=fit_thresholds(y,p);scores=metric(y,p,thresholds)
        if scores['macro_f1']>best+1e-10:best=scores['macro_f1'];winner=(members,np.mean(tests,0),thresholds,scores)
    if winner:
        members,p,thresholds,scores=winner;path=project/'bonus/ensemble.zip'
        _,sha=write_submission(test_ids,predict_labels(p,thresholds,allowed),path)
        base={'name':'bonus_ensemble','submission':str(path),'dev_metrics':scores,'thresholds':thresholds,'prediction_sha256':sha,
            'kind':'development_selected_ensemble','members':[r['name'] for r in members],'counts_as_new_adaptation_method':False}
    atomic_json(project/'recommended_submission.json',base);return base

def export_manifest(project):
    project=Path(project).resolve();catalog=read_json(project/'catalog.json');entries=[]
    for item in catalog:
        row={k:item[k] for k in ('name','architecture','method','calibration','dev_metrics')}
        for key in ('warm_checkpoint','adapted_checkpoint'):
            path=Path(item[key]).resolve();saved=torch.load(path,map_location='cpu',weights_only=False)
            parent=Path(saved['config']['parent']).resolve()
            row[key]={'relative_path':str(path.relative_to(project)),'sha256':file_hash(path),
                'parent_relative_path':str(parent.relative_to(project)),'parent_sha256':file_hash(parent)}
        row['features_relative_path']=str(Path(item['features']).resolve().relative_to(project));entries.append(row)
    atomic_json(project/'phase5_model_manifest.json',{'format_version':1,'models':entries,'tokenizer':'parents/tokenizer.json',
        'restoration':'Resolve checkpoint/parent paths under extracted capstone_phase5; restore adapted head_init from warm_checkpoint; fusion uses full warm and selected-evidence adapted logits. Lexical artifacts are in lexical/.'})
    return project/'phase5_model_manifest.json'

def bundle_models(project,destination):
    """Portable learned artifacts; raw official data/feature banks and optimizer recovery are reproducible."""
    project=Path(project).resolve();destination=Path(destination).resolve();temporary=destination.with_suffix('.tmp')
    selected=[]
    for path in project.rglob('*'):
        if not path.is_file() or path.resolve() in (destination,temporary):continue
        relative=path.relative_to(project)
        if relative.parts[0] in ('data','token_cache','preflight'):continue
        if path.name in ('features.f16','done.u8','latest.pt'):continue
        if path.suffix in ('.zip','.tmp') or '__pycache__' in relative.parts:continue
        if path.suffix=='.pt' and path.name not in ('best.pt','decoder.pt','encoder_decoder.pt','local_gqa.pt','hybrid.pt'):continue
        selected.append(path)
    estimated=sum(p.stat().st_size for p in selected)
    if shutil.disk_usage(destination.parent).free<estimated+128*1024**2:raise RuntimeError('Not enough space for model archive. Download submissions first; keep parents/ and best.pt files.')
    with zipfile.ZipFile(temporary,'w',zipfile.ZIP_DEFLATED,compresslevel=1) as archive:
        for path in selected:archive.write(path,Path(project.name)/path.relative_to(project))
    temporary.replace(destination);return str(destination)
'''
(CODE/'lsi_workflow.py').write_text(MODULE_SOURCE.lstrip('\n'),encoding='utf-8')
print('Wrote','lsi_workflow.py')

In [ ]:
MODULE_SOURCE=r'''
"""Optional supplied test truth: final reporting only, never imported by training workers."""
from pathlib import Path
import json,zipfile
from lsi_data import records,read_json,atomic_json,target_matrix,metric

def evaluate_supplied_test(root,submission,output=None):
    raw=read_json(Path(root)/'Test/test.json')
    try:truth=records(raw,'dev')
    except ValueError as error:
        if 'labels must be a list' in str(error):return {'available':False,'note':'No valid supplied test truth. No test score was computed.'}
        raise
    with zipfile.ZipFile(submission) as archive:pred=json.loads(archive.read('prediction.json'))
    byid={r['id']:r['pred'] for r in pred}
    if len(byid)!=len(pred) or set(byid)!={r['id'] for r in truth}:raise ValueError('Test truth/prediction IDs differ.')
    dimension=max((j for r in truth for j in r['gold']),default=-1)+1
    if dimension<1:raise ValueError('Supplied test truth has no labels.')
    if any(type(j) is not int or j<0 or j>=dimension for row in byid.values() for j in row):raise ValueError('Prediction label outside supplied evaluator truth dimension.')
    y=target_matrix(truth,dimension);p=target_matrix([{'gold':byid[r['id']]} for r in truth],dimension)
    scores={'available':True,**metric(y,p),'evaluation_only':True,'submission':str(submission),
        'note':'Frozen predictions scored after all development-based choices. No test-driven model, threshold, fusion or submission selection.'}
    if output:atomic_json(output,scores)
    return scores
'''
(CODE/'lsi_evaluate_test.py').write_text(MODULE_SOURCE.lstrip('\n'),encoding='utf-8')
print('Wrote','lsi_evaluate_test.py')

In [ ]:
MODULE_SOURCE=r'''
#!/usr/bin/env python3
"""Local evaluator for Phase 5.

Truth JSON: a list of objects, [{"id":"...", "gold":[0,2,5,...]}, ...]
Prediction JSON: [{"id":"...", "pred":[0,2,5,...]}, ...]
"""
import argparse,json,sys
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def read_truth(path):
    try:
        x=json.load(open(path,encoding="utf-8"))
    except json.JSONDecodeError as e:
        raise ValueError(f"Truth must be a JSON file containing a list of records: {e}")
    if not isinstance(x,list) or not x: raise ValueError('Truth must be a non-empty JSON list.')
    out={}; max_label=-1
    for n,r in enumerate(x):
        if not isinstance(r,dict) or 'id' not in r or 'gold' not in r: raise ValueError(f'Truth item {n} must contain id and gold.')
        if not isinstance(r['id'],str) or not r['id'].strip() or not isinstance(r['gold'],list): raise ValueError(f'Invalid truth item {n}.')
        if r['id'] in out: raise ValueError(f'Duplicate truth ID: {r["id"]}')
        seen=set()
        for label in r['gold']:
            if isinstance(label,bool) or not isinstance(label,int) or label<0 or label in seen: raise ValueError(f'Invalid/duplicate label in truth item {n}.')
            seen.add(label); max_label=max(max_label,label)
        out[r['id']]=r['gold']
    return out,max_label+1

def read_pred(path,nlab):
    x=json.load(open(path,encoding='utf-8'))
    if not isinstance(x,list) or not x: raise ValueError('Prediction must be a non-empty JSON list.')
    out={}
    for n,r in enumerate(x):
        if not isinstance(r,dict) or 'id' not in r or 'pred' not in r: raise ValueError(f'Prediction item {n} must contain id and pred.')
        if not isinstance(r['id'],str) or not isinstance(r['pred'],list) or r['id'] in out: raise ValueError(f'Invalid or duplicate prediction item {n}.')
        seen=set()
        for x in r['pred']:
            if isinstance(x,bool) or not isinstance(x,int) or x<0 or x>=nlab or x in seen: raise ValueError(f'Invalid/duplicate label in prediction item {n}.')
            seen.add(x)
        out[r['id']]=r['pred']
    return out

def main():
    ap=argparse.ArgumentParser(description='Evaluate Phase 5 multilabel predictions on your own held-out split.')
    ap.add_argument('--truth',required=True); ap.add_argument('--prediction',required=True); a=ap.parse_args()
    t,n=read_truth(a.truth); p=read_pred(a.prediction,n)
    if set(t)!=set(p): raise ValueError(f"ID mismatch: missing={len(set(t)-set(p))}, extra={len(set(p)-set(t))}")
    yt=np.zeros((len(t),n),dtype=np.uint8); yp=np.zeros((len(t),n),dtype=np.uint8)
    for i,k in enumerate(t):
        for x in t[k]: yt[i,x]=1
        for x in p[k]: yp[i,x]=1
    print(f'Examples: {len(t)}'); print(f'Labels: {n}')
    print(f'Macro-F1: {f1_score(yt,yp,average="macro",zero_division=0):.6f}')
    micro=f1_score(yt,yp,average='micro',zero_division=0)
    subset=accuracy_score(yt,yp)
    print(f'Micro-F1: {micro:.6f}')
    print(f'Subset Accuracy: {subset:.6f}')
if __name__=='__main__':
    try: main()
    except Exception as e: print(f'EVALUATION FAILED: {e}',file=sys.stderr); sys.exit(1)
'''
(CODE/'evaluate_phase5_official.py').write_text(MODULE_SOURCE.lstrip('\n'),encoding='utf-8')
print('Wrote','evaluate_phase5_official.py')

In [ ]:
importlib.invalidate_caches()
for name in ('lsi_workflow','lsi_run','lsi_model','lsi_core','lsi_data','lsi_evaluate_test','legal_models'):
    sys.modules.pop(name,None)
from lsi_data import prepare_data,read_json,atomic_json,digest,target_matrix,predict_labels,metric
from lsi_workflow import (restore_phase1,obtain_data,experiments,benchmark,smoke_all,lexical_branch,
                          run_all,bonus_ensemble,export_manifest,bundle_models,bundle_working)
design={'settings':SETTINGS,'use_lexical':USE_LEXICAL_FUSION,'lexical_max_features':LEXICAL_MAX_FEATURES,
        'input_root':str(PHASE1_ROOT),'parent_overrides':{k:str(v) for k,v in PARENT_OVERRIDES.items()},'pipeline_version':1}
design_path=PROJECT/'training_design.json'
if design_path.exists() and read_json(design_path)!=design:
    raise ValueError('Training design changed. Restore earlier settings or use a new PROJECT; completed artifacts remain in place.')
atomic_json(design_path,design)
parents,tokenizer_path=restore_phase1(PHASE1_ROOT,PROJECT/'parents',PARENT_OVERRIDES)
for architecture,path in parents.items():
    info=torch.load(path,map_location='cpu',weights_only=False)
    required=SETTINGS['chunk_tokens']+(16 if architecture=='decoder' else 0)
    if required>info['config']['max_positions']:
        raise ValueError(f'{architecture}: chunk plus prompt exceeds original positional capacity; reduce chunk_tokens.')
    del info
print(json.dumps(read_json(PROJECT/'parents/input_manifest.json'),indent=2))

## Official data, case audit and full token coverage

The full document is encoded once for the frozen branch. The adapted branch uses first/last/statute-cue windows with original positions. The label vocabulary is derived from public train+dev only. Test labels cannot become model targets. Exact source-case overlap and normalized training-text copies of held-out documents are removed before fitting.

Feature sizes below estimate storage, not completion time. Keep the Kaggle output disk budget in mind when choosing optional full-working archives; individual submission ZIPs are small.

In [ ]:
data_root=obtain_data(PROJECT,DATA_ROOT,DATA_REVISION)
DATA_PATH=prepare_data(data_root,tokenizer_path,PROJECT/'token_cache',SETTINGS['chunk_tokens'],SETTINGS['overlap'])
data=read_json(DATA_PATH)
assert all(data['docs'][i]['gold'] is None for i in data['splits']['test'])
assert set(data['splits']['all'])==set(data['splits']['train'])|set(data['splits']['dev'])
assert set(data['splits']['all']).isdisjoint(data['splits']['test'])
print(json.dumps(data['audit'],indent=2))
print('Tokens:',data['total_tokens'],'windows:',data['total_chunks'],'label dimension:',data['label_count'])
print('Free working disk GiB:',round(shutil.disk_usage(WORKING).free/1024**3,2))
atomic_json(PROJECT/'data_audit.json',data['audit'])
rows=experiments(parents,SETTINGS,data)
for row in rows:print(row['name'],row['architecture'],row['method'],row['head_type'],'adaptation view:',row['view'])

## Measured preflight

The benchmark is a rough encoder-adaptation forecast **only**. It excludes full-document feature extraction, head training, dev evaluation, KD caching, DDP startup and I/O. The GPU smoke test then performs one actual two-process update per method. Its synthetic teacher is limited to this preflight and is rejected by real KD training.

The smoke test must pass all seven methods. A partial forecast must not be interpreted as total notebook runtime.

In [ ]:
if RUN_BENCHMARK:
    measurements=benchmark(rows,DATA_PATH)
    atomic_json(PROJECT/'preflight_measurements.json',measurements)
    print('Rough adaptation-only forecast:',round(sum(r['rough_adaptation_hours'] for r in measurements),2),'hours')
    print('Additional stages are excluded; this is not a total-runtime estimate.')
if RUN_GPU_SMOKE:smoke_all(PROJECT,rows,DATA_PATH,SETTINGS)

## Train-only lexical branch and seven neural adaptations

TF-IDF and each lexical classifier fit train only. Neural heads compare ASL against weighted BCE on reusable complete-document features; the better dev loss candidate initializes adaptation. Cached heads are shared when their foundation and head type match.

Each adapted model is genuinely trained, including the mandatory LoRMA+ and smaller KD student. Dev chooses the combination of full-document, adapted-evidence and optional lexical probabilities, then thresholds. Adaptation always has nonzero weight. A weight of 1.0 means evidence-only inference is selected. Completed candidates save their ZIP immediately. Test truth is not used for any of these choices.

In [ ]:
lexical=lexical_branch(PROJECT,data_root,DATA_PATH,LEXICAL_MAX_FEATURES) if USE_LEXICAL_FUSION else None
catalog=run_all(PROJECT,rows,DATA_PATH,SETTINGS,lexical)
if MAKE_BONUS_ENSEMBLE:
    recommendation=bonus_ensemble(PROJECT,DATA_PATH,lexical)
else:
    best=max(catalog,key=lambda item:item['dev_metrics']['macro_f1'])
    recommendation={'name':best['name'],'submission':best['submission'],'dev_metrics':best['dev_metrics'],'kind':'single_adaptation'}
    atomic_json(PROJECT/'recommended_submission.json',recommendation)
print('Development-selected recommendation:',recommendation['submission'])
print('Portable model manifest:',export_manifest(PROJECT))

## Official evaluator parity and actual measured results

The official script uses every integer column through the largest reference label, including ID gaps. Dev exports respect that reference dimension, while the test label vocabulary remains the public train+dev vocabulary. The table contains actual results from your run, not paper scores. These dev scores were used to select losses, epochs, fusion and thresholds.

In [ ]:
catalog=read_json(PROJECT/'catalog.json')
comparison=[]
expected=[data['docs'][i]['id'] for i in data['splits']['test']]
allowed={j for j,n in enumerate(data['train_support']) if n>0}
for item in catalog:
    result=subprocess.run([sys.executable,str(CODE/'evaluate_phase5_official.py'),
        '--truth',str(PROJECT/'dev_truth.json'),'--prediction',str(PROJECT/'runs'/item['name']/'dev_prediction.json')],
        text=True,capture_output=True,check=True)
    print(item['name'],result.stdout.strip().replace('\n',' | '))
    for label,key in [('Macro-F1','macro_f1'),('Micro-F1','micro_f1'),('Subset Accuracy','subset_accuracy')]:
        measured=float(next(line.split(':')[1] for line in result.stdout.splitlines() if line.startswith(label+':')))
        assert abs(measured-item['dev_metrics'][key])<1e-6,(item['name'],key)
    with zipfile.ZipFile(item['submission']) as archive:
        assert archive.namelist()==['prediction.json']
        pred=json.loads(archive.read('prediction.json'))
    assert [r['id'] for r in pred]==expected and len(expected)==len(set(expected))
    assert all(isinstance(r['pred'],list) and all(type(j) is int and j in allowed for j in r['pred'])
               and len(r['pred'])==len(set(r['pred'])) for r in pred)
    cal=item['calibration'];lex_weight=cal['lexical_weight']
    comparison.append({'name':item['name'],'architecture':item['architecture'],'method':item['method'],
        **item['dev_metrics'],'selected_loss':item['selected_loss'],
        'adapted_total_parameters':item['parameters']['total'],'adapted_trainable_parameters':item['parameters']['trainable'],
        'warm_total_parameters':item['warm_parameters']['total'],
        'warm_probability_weight':(1-cal['adapt_weight'])*(1-lex_weight),
        'adapted_probability_weight':cal['adapt_weight']*(1-lex_weight),'lexical_weight':lex_weight,
        'prediction_sha256':item['prediction_sha256']})
with (PROJECT/'comparison.csv').open('w',newline='') as handle:
    writer=csv.DictWriter(handle,fieldnames=list(comparison[0]));writer.writeheader();writer.writerows(comparison)
print('Distinct prediction sets:',len({r['prediction_sha256'] for r in catalog}),'of 7 trained methods')
print('Submission-count and mandatory-method checklist:',json.dumps(read_json(PROJECT/'required_submission_plan.json'),indent=2))
display(FileLink(str(PROJECT/'comparison.csv')))

## Rare-statute diagnostics

Per-label development support and errors help distinguish an imbalance problem from a label with no training examples. Scores over real ID gaps remain included in the official Macro-F1; the diagnostic file also identifies which labels actually occurred publicly. Attention weights indicate model focus, not causal legal explanations.

In [ ]:
from sklearn.metrics import precision_recall_fscore_support
truth_rows=read_json(PROJECT/'dev_truth.json')
y=target_matrix(truth_rows,data['label_count']).astype(np.uint8)
label_rows=[]
for item in catalog:
    prediction=read_json(PROJECT/'runs'/item['name']/'dev_prediction.json')
    byid={r['id']:r['pred'] for r in prediction}
    yp=target_matrix([{'gold':byid[r['id']]} for r in truth_rows],data['label_count']).astype(np.uint8)
    precision,recall,f1,support=precision_recall_fscore_support(y,yp,average=None,zero_division=0)
    for j in data['label_ids']:
        label_rows.append({'method':item['name'],'statute_id':j,'train_support':data['train_support'][j],
                          'dev_support':int(support[j]),'precision':float(precision[j]),'recall':float(recall[j]),'f1':float(f1[j])})
with (PROJECT/'label_analysis.csv').open('w',newline='') as handle:
    writer=csv.DictWriter(handle,fieldnames=list(label_rows[0]));writer.writeheader();writer.writerows(label_rows)
import matplotlib.pyplot as plt
fig,axes=plt.subplots(1,2,figsize=(13,4))
axes[0].barh([r['name'] for r in comparison],[r['macro_f1'] for r in comparison])
axes[0].set_xlim(0,1);axes[0].set_xlabel('Development Macro-F1')
counts=np.asarray(data['train_support']);counts=counts[counts>0]
axes[1].hist(counts,bins=min(30,max(1,len(counts))));axes[1].set_yscale('log')
axes[1].set_xlabel('Training positives per observed statute');axes[1].set_ylabel('Statute count (log)')
fig.tight_layout();fig.savefig(PROJECT/'measured_results.png',dpi=150);plt.show()
display(FileLink(str(PROJECT/'label_analysis.csv')))

## Optional supplied test truth: reporting only

This runs after all seven submissions and the development-based recommendation are fixed. It reports supplied labels when available and skips scoring if the official test JSON has no labels. Set the flag false to skip this reporting step. The reporter does not alter the catalog, thresholds, models, recommendation or ZIPs.

In [ ]:
if EVALUATE_SUPPLIED_TEST_LABELS:
    from lsi_evaluate_test import evaluate_supplied_test
    test_reports=[]
    for item in catalog:
        report=evaluate_supplied_test(data_root,item['submission'],PROJECT/'test_evaluation'/f"{item['name']}.json")
        test_reports.append({'name':item['name'],**report})
        print(item['name'],report)
    atomic_json(PROJECT/'test_evaluation/results.json',test_reports)

## Download submissions and learned models

Each individual ZIP below contains a root `prediction.json` with one original test ID and a unique integer statute list per case. The collection ZIP contains those seven separate submission ZIPs. Submit the individual experiment ZIPs to Codabench.

The model archive retains learned checkpoints, dense Phase-1 parents, source, calibration and lexical artifacts. Large raw/token/feature banks and optimizer recovery are omitted from that compact download; they remain in the Kaggle working output. The optional full-working archive includes those files too and may need many additional GB of free disk space.

In [ ]:
collection=WORKING/'phase5_seven_submissions.zip'
with zipfile.ZipFile(collection,'w',zipfile.ZIP_STORED) as archive:
    for item in catalog:archive.write(item['submission'],Path(item['submission']).name)
for item in catalog:display(FileLink(item['submission']))
display(FileLink(str(collection)))
selected=read_json(PROJECT/'recommended_submission.json')
print('Development-selected score candidate:',selected['submission'])
display(FileLink(selected['submission']))
if MAKE_MODEL_ARCHIVE:
    portable=bundle_models(PROJECT,WORKING/'phase5_models_and_code.zip')
    display(FileLink(portable))
if MAKE_FULL_WORKING_ARCHIVE:
    sizes=sum(p.stat().st_size for p in WORKING.rglob('*') if p.is_file() and p.name!='phase5_all_working.zip')
    if shutil.disk_usage(WORKING).free<sizes+128*1024**2:
        print('Full-working ZIP needs more free space. Individual submission and model downloads above are available.')
    else:
        full=bundle_working(WORKING,WORKING/'phase5_all_working.zip')
        display(FileLink(full))

# CS781 Phase 5 — Legal Statute Identification

This notebook builds **seven trained adaptation submissions** for multi-label statute identification on **Kaggle T4 ×2**. Its selection objective is **Macro-F1**, with Micro-F1 and Subset Accuracy checked using the supplied official evaluator. Research motivates the candidates; the best candidate on this particular dataset can only be established by running your models. No competition score or total runtime is promised.

The [current capstone brief](https://docs.google.com/document/d/1olqoT5KPhf5CSJLBNjXGyKyyyQDBr6jcnWqPUh5cNk4/edit) gives the Phase-5 deadline as **October 26, 2026, 11:59 PM IST**. It requires all four original Phase-1 families, LoRMA and knowledge distillation. Its general instructions also require at least six distinct submissions in Phases 2–7; identical/repeated submissions do not count. This is AI-assisted code; follow the brief's disclosure and course-use requirements. Nothing here uploads to Codabench.

## Input folders

Attach your original `phase-1-outputs` Kaggle dataset. The default root is your previously supplied path:

```text
/kaggle/input/datasets/hardik4tiwari/phase-1-outputs/capstone_phase1/
    tokenizer.json
    runs/
        main_decoder/best_model.pt
        main_encoder_decoder/best_model.pt
        main_local_gqa/best_model.pt
        main_hybrid/best_model.pt
```

Change `PHASE1_ROOT` only if your mount differs. `PARENT_OVERRIDES` can point individual architectures to their original Phase-1 files. Checkpoint origin, tokenizer hash, architecture, full state dictionary and 50M–300M parent size are validated. The smaller distilled student is constructed from its original Phase-1 decoder; it is allowed to have fewer parameters than its teacher.

**Phase-4 task checkpoints do not replace these parents.** The Phase-5 brief explicitly says every experiment starts from Phase 1. Your Phase-2/3/4 predictions and task-adapted weights are not used to initialize this run. The notebook reads `capstone_phase1` directly, without scanning `all_kaggle_outputs` or extracting unrelated archives.

Use the official data:

```text
Phase_5_LSI/
    Train/train.json
    Train/dev.json
    Test/test.json
```

Set `DATA_ROOT` to that directory or its parent when attached as a Kaggle input. With `DATA_ROOT=None`, the notebook downloads these three files from the official [Hugging Face repository](https://huggingface.co/datasets/Exploration-Lab/CS-781-Capstone), resolves a commit once and records SHA-256 hashes. Accept the repository's terms and enable a Kaggle secret named `HF_TOKEN` when required. Do not paste tokens into the notebook. A complete verified local copy resumes without another network download.

## Architecture and score strategy

The trained input remains your own Phase-1 language model. No external model weights, external embeddings, external legal corpus or LLM/API predictions are downloaded.

A full-document branch encodes every overlapping window with the frozen foundation. Each window stores the mean and final valid token representations. Six positional values describe where it occurred. A small BiGRU or Transformer then contextualizes these window representations. **Each statute has its own attention query and output vector**, so different labels can attend to different evidence. Global mean/max context and a gated maximum chunk score provide additional document and multiple-instance signals.

A second branch adapts the foundation and head on a small number of evidence windows. The default is **three windows per document**: first, last and the strongest remaining window containing generic statute cues such as “section”, “article”, “act” or “code”. Selection uses document text, never gold labels. If a document has fewer windows, all available windows are used. The learned positional coordinates retain the selected windows' original locations.

The two branches share compatible head initialization. They are trained and cached separately; frozen features are not passed off as updated encoder features. Their probabilities are combined by a development-selected weight. The grid includes adaptation weights 0.25, 0.5, 0.75 and 1.0. **A weight of 1.0 explicitly selects the adapted evidence-only branch.** The catalog records whether the final prediction uses the full-document branch. Every original token is still represented in the frozen feature bank available to selection. This is a custom two-view design, not a claim to reproduce a published model exactly or to equal seven full-document end-to-end fine-tunes.

The optional train-only lexical component fits TF-IDF and independent logistic SGD classifiers. It can help with explicit statute citations and rare word/number combinations. Its vocabulary and classifiers fit train only. Dev chooses a fusion weight from 0, 0.2 or 0.4; neural predictions always retain at least 60% of the probability mixture. The adapted neural component therefore always contributes to each required experiment.

## Seven experiments

| Submission | Original family | Adaptation | Context head |
|---|---|---|---|
| `01_decoder_full.zip` | Decoder | Full fine-tuning | BiGRU + label-wise attention |
| `02_encoder_decoder_partial.zip` | Encoder–decoder | Final decoder layers + final norm | Chunk Transformer + label-wise attention |
| `03_local_gqa_lorma.zip` | Local/GQA | LoRMA+ multiplicative updates | BiGRU + label-wise attention |
| `04_hybrid_adapter.zip` | LSTM–Transformer hybrid | Residual bottleneck adapters | BiGRU + label-wise attention |
| `05_decoder_prompt.zip` | Decoder | Trainable soft input prompts | BiGRU + label-wise attention |
| `06_encoder_decoder_gradual.zip` | Encoder–decoder | Gradual layer unfreezing | BiGRU + label-wise attention |
| `07_decoder_student_kd.zip` | Smaller original-decoder student | Hard labels + Bernoulli KD | BiGRU + label-wise attention |

Loss and threshold variants optimize these experiments; they are not counted as additional adaptation techniques. All four families and the mandatory LoRMA and KD experiments are present. The four best families may omit the KD candidate when another decoder is stronger. Therefore `required_submission_plan.json` includes the KD ZIP as well as family and LoRMA coverage. Submit the seven individual candidate ZIPs and verify that at least six prediction sets are distinct. Merely running different methods does not establish the distinct-submission count. An optional score ensemble is an additional candidate and is not counted as another fine-tuning technique.

## Loss and rare-label handling

Cached head training compares **asymmetric loss (ASL)** with **weighted binary cross-entropy**. The latter uses a bounded square-root inverse-frequency positive weight. The stronger dev Macro-F1 head initializes the single encoder-adaptation run for that method, and its loss is retained. This adds inexpensive head comparisons instead of doubling encoder training. `head_losses` controls the candidates.

ASL uses independent sigmoid probabilities, positive focusing exponent 0, negative exponent 4 and negative-probability clipping 0.05. Loss calculations run in FP32. There is no softmax competition between statutes: several labels can be positive simultaneously. The model bias starts from smoothed training prevalence.

The label space comes from **train+dev only**, not supplied test labels. Integer IDs are never renumbered. The official evaluator uses columns `0 … max(reference ID)` inclusive, including unused ID gaps; local metrics follow that convention. Labels without any train positives are reported and suppressed in predictions. Integer IDs alone do not provide a meaningful zero-shot statute description, so no unseen-label capability is claimed.

Dev chooses a global threshold, then compares a per-label adjustment. A label needs at least three dev positives to receive its own threshold; its estimate is shrunk toward the global threshold using support/(support+20). The global solution is retained if shrinkage does not improve Macro-F1. Test labels are not involved. Empty prediction lists are accepted by the provided evaluator, so an arbitrary top-one label is not forced into an uncertain prediction.

## Mandatory distillation

After the six larger models finish, the strongest dev-selected Phase-5 model is the teacher. Its full-view/evidence-view/fusion probabilities on **training documents only** become independent per-label logits. No Phase-3 rhetorical-role logits or binary Phase-4 logits are used.

The student retains four uniformly spaced blocks from the original eight-block Phase-1 decoder by default. Change `student_layers` if your original decoder has a different depth; it must remain smaller. The loss combines hard-label supervision with temperature-2 Bernoulli KL, multiplied by temperature squared. Both positive and negative Bernoulli outcomes are included. This does not incorrectly normalize all statute labels with a softmax. Teacher origin, dataset fingerprint, split, label dimension and larger parameter count are checked. Synthetic zero teachers are accepted only for the separate GPU preflight, not real training.

## Why the runtime is lower

1. Full-document feature extraction is done once per distinct foundation representation: four full-sized families and the pruned student. Decoder methods reuse the same frozen bank and compatible warm head. Encoder–decoder heads reuse their bank even when their context heads differ.
2. Head epochs train on disk-backed FP16 feature arrays. Encoder adaptation processes selected evidence instead of every window in every epoch. There is no second full-corpus re-encoding for each adapted model.
3. Feature extraction groups windows from several short cases into encoder batches. Both GPUs participate in feature extraction and adaptation through DDP. Small cached heads use one GPU.
4. FP16 autocast, gradient scaling, fused CUDA AdamW, accumulation with `no_sync`, length grouping and native PyTorch attention/recurrent kernels reduce overhead. No unsupported BF16 requirement or FlashAttention-2 package is imposed on T4.
5. Activation checkpointing reduces saved activations. For the hybrid, the fused LSTM stays outside recomputation while Transformer blocks are checkpointed. Each replay restores its own padding mask. This incorporates the earlier Phase-4 repair.
6. Completed banks, heads, predictions and optimizer/RNG recovery files are reused. A completed candidate's ZIP is written immediately.

Checkpointing trades additional computation for memory; it is not automatically faster. Local/GQA uses portable dense SDPA masks and repeated KV heads where necessary, not a custom block-sparse kernel. Parameter-efficient updates do not remove the foundation's forward pass.

The embedded benchmark reports an **adaptation-only compute forecast** from measured work on your GPU. It excludes full-document feature extraction, head training, validation, teacher-cache creation, DDP startup and I/O. Treat it as partial information, not a total completion-time promise. Feature-bank size and free disk space are reported before training. No elapsed-time budget silently cuts epochs or labels.

## Data and test-label isolation

Record lists, a single record, ID mappings, column exports and keyed/pandas-style columns are supported. Text can be a string or ordered string-sentence list. Gold statute sets must contain nonnegative integers. Duplicated label IDs within a set are removed. Missing IDs and malformed labels fail before training.

Repeated source IDs are handled at case level: exact copies collapse; distinct texts concatenate in original order; supplied train/dev label sets are unioned. Every output retains one exact original case ID. The audit reports this aggregation. Training source cases found in dev/test are removed, as are exact normalized training-text copies of held-out documents. Dev source cases also found in test are excluded from selection. No invented test IDs are used.

**If test labels are supplied, the inference parser discards them unconditionally.** They cannot enlarge the model vocabulary or enter tensors used for fitting. `train` contains only training cases. Dev is used for checkpoint, loss, fusion and threshold choices; no automatic train+dev refit is performed. Test feature extraction is a fixed encoder forward pass without fitting.

The final test reporter is enabled by default to accommodate the assignment files when they contain supplied test labels. It skips scoring if labels are absent, and reads supplied truth only after all predictions and dev-selected choices are finalized. It writes a separate report and never changes models, thresholds, recommended submissions or ZIPs. Private reference labels are not needed to run the notebook.

## Research checked and implementation decisions

| Primary source | Finding used | Implementation and limit |
|---|---|---|
| [Chalkidis et al., ACL 2019](https://aclanthology.org/P19-1636/) | Legal multi-label classification benefits from label-wise contextual attention; document zones merit comparison. | One attention query per statute over contextualized window embeddings, plus an evidence-view comparison. EURLEX results are not a score forecast for this dataset. |
| [Mullenbach et al., NAACL 2018, CAML](https://aclanthology.org/N18-1100/) | Labels can attend to different portions of a long document. | Statute-specific evidence pooling. The backbone is your own foundation, not CAML's CNN or external embeddings. Attention is a diagnostic rather than a causal explanation. |
| [Ridnik et al., ICCV 2021, ASL](https://arxiv.org/html/2009.14119v4) | Separate positive/negative focusing and suppress easy negatives in multi-label loss. | FP32 ASL compared with weighted BCE on dev. Image-benchmark gains are not assumed to transfer automatically to statutes. |
| [Bihany, Patel & Modi, Findings ACL 2025, LoRMA](https://aclanthology.org/2025.findings-acl.527.pdf) | Multiplicative adaptation with rank inflation and efficient operation ordering. | **LoRMA+ equation 8:** `y = W x; output = y + (alpha/r) B(Ay)`. B starts at zero. No large square product is constructed. This is the additive-identity inflation variant, not permutation inflation. |
| [Houlsby et al., 2019](https://arxiv.org/abs/1902.00751) | Small adapter modules can adapt a frozen backbone. | Post-block residual bottlenecks on the hybrid. This is an adapter variant, not an exact replication of the paper's placement. |
| [Lester et al., EMNLP 2021](https://aclanthology.org/2021.emnlp-main.243/) | Continuous learned prompts condition frozen language models. | Learned prefix input embeddings initialized from your own token embeddings. It is soft prompt tuning, not per-layer K/V prefix tuning; the paper's large-model results need not hold at 50M parameters. |
| [Hinton et al., 2015](https://arxiv.org/abs/1503.02531) | Temperature-softened predictions transfer task information into a smaller model. | Independent Bernoulli KL for multi-label targets, using a real trained Phase-5 teacher. |
| [Lipton et al., 2014](https://arxiv.org/abs/1402.1892) | F1-optimal decision thresholds need not equal 0.5. | Dev-only global and support-shrunk label thresholds. Shrinkage is an engineering choice here; the paper does not prove this specific rule optimal. |
| [Chen et al., 2016](https://arxiv.org/abs/1604.06174), [PyTorch AMP examples](https://docs.pytorch.org/docs/stable/notes/amp_examples.html) | Saved activations trade off against recomputation; mixed precision requires appropriate scaling. | Non-reentrant checkpoints and FP16/GradScaler. GPU memory/runtime are measured locally rather than inferred from paper hardware. |

The full-document/evidence mixture, head sharing, bounded lexical fusion and loss-selection procedure are this notebook's combined design. No paper establishes it as the best architecture for the competition.

## Run and outputs

Choose **GPU T4 ×2**, attach the Phase-1 dataset, enable Internet only if downloading official data, edit input paths, then run top to bottom. The GPU preflight must complete all seven actual distributed optimizer updates before long training begins. Previous duplicate-ID and supplied-test-label fixes are included; no repair cells are required.

The notebook writes under `/kaggle/working/capstone_phase5/`:

```text
parents/                         verified own Phase-1 weights and tokenizer
code/                            complete runtime and original official evaluator
token_cache/                     complete token data and audited manifest
features/                        reusable full-document banks and cached heads
runs/<method>/                   adapted best.pt, recovery state and predictions
submissions/<method>.zip         seven individual prediction.json archives
catalog.json                     methods, measured dev results, losses and fusion
best_four_architectures.json      strongest dev candidate per original family
required_submission_plan.json     family coverage plus mandatory LoRMA/KD
recommended_submission.json      development-selected score candidate
phase5_model_manifest.json        relative checkpoint/dependency locations
comparison.csv                   actual dev metrics and parameter counts
label_analysis.csv               actual per-label support and dev F1
```

Each individual submission ZIP contains only a root `prediction.json`:

```json
[{"id":"original_case_id","pred":[1,7,23]}]
```

The final cells create a seven-submission collection ZIP and a model/code archive. The latter omits recomputable raw/token/feature banks and optimizer recovery, but retains learned `best.pt` checkpoints, dense parents, calibration, lexical artifacts and source. Whole-document features can be rebuilt from the official input data. Kaggle's saved working output still contains the complete project. An optional full-working archive switch is also available; it can require substantial extra disk space and time.

Keep `PROJECT`, paths, training settings and GPU count fixed for exact recovery. Input, parent and configuration fingerprints guard cache reuse. Changing the training design requires a new project; an existing compatible completed run is reused. Different adaptation methods may legitimately produce identical predicted label sets. The code reports their hashes and flags when fewer than six base prediction sets are distinct. In that case improve genuine experiments using dev evidence; the count requirement is not yet satisfied. It does not manufacture uniqueness by changing labels.

Validation performed here uses small **synthetic CPU fixtures**: all seven end-to-end runs; ASL/BCE head comparison; trainable/frozen gradients; prompt gradients; padding/checkpoint parity; real teacher/student training; original official evaluator parity; label-ID gaps; duplicate aggregation; test-label isolation; and ZIP validation. Actual T4/CUDA throughput and leaderboard performance cannot be tested here. The in-notebook GPU preflight and dev evaluation are therefore required checks on your Kaggle run.